**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# AIMS - Vero XEL, ricevitori e rivalutazione Sim2Real

Pipeline dedicata dopo il controllo incrociato dei log SRT e dei file di calibrazione.


In [ ]:
from pathlib import Path
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV, HuberRegressor, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SEQ_LEN = 60                 # 60 two-minute samples span approximately two hours.
EXPECTED_STEP_SEC = 120      
MIN_STEP_SEC = 60             
MAX_GAP_SEC = 180            
MAX_SENSOR_RATE_C = 5.0       
USE_RESAMPLED_WINDOWS = True  
TARGET = "XEl Offset"        
SPEC_ARCSEC = 5.0             # Historical reference threshold; receiver-specific requirements differ.
INCLUDE_CURRENT_TEMP = True  


PROJECT_SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents, Path("/content/metrologyml"), Path.home() / "Downloads"]
DATA_SUBDIRS = [Path("."), Path("data/raw/real"), Path("data/raw/real/optional"), Path("data/raw/control"), Path("data/raw/fem"), Path("data/diagnostics/control"), Path("data/diagnostics/fem"), Path("data/diagnostics/history")]
DATA_DIRS = list(dict.fromkeys(root / subdir for root in PROJECT_SEARCH_ROOTS for subdir in DATA_SUBDIRS))
REAL_CANDIDATES = ["EffectiveDataset_Tseries_with_azimuth.csv", "EffectiveDataset_Tseries_total.csv", "EffectiveDataset_Tseries (2).xlsx", "EffectiveDataset_Tseries.xlsx", "EffectiveDataset_Tseries.csv"]
FEM_CANDIDATES = ["Dataset (1).xlsx", "dataset_multistep_jul_aug.csv", "Dataset.xlsx"]
POINTING_CANDIDATES = ["pointing_offsets_real_azimuth.csv"]
USE_POINTING_TARGETS_WITH_AZIMUTH = False  
MAX_ABS_XEL_FOR_TRAIN = 60.0    
MAX_ABS_EL_FOR_TRAIN = 180.0    
FILTER_MAINTENANCE_BY_NAME = False  # Maintenance scans are not rejected solely on their project label.
POINTING_CLEANING_MODE = "extreme_only"  
STABLE_JUMP_WINDOW_MIN = 180
STABLE_DXEL_JUMP_ARCSEC = 15.0
STABLE_DEL_JUMP_ARCSEC = 20.0

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


In [ ]:
def find_data_file(candidates, data_dirs=DATA_DIRS):
    for d in data_dirs:
        for name in candidates:
            p = d / name
            if p.exists():
                return p
    raise FileNotFoundError(f"No file found among: {candidates}")


def read_table(path: Path) -> pd.DataFrame:
    """Read CSV/XLSX, with a direct openpyxl fallback for version conflicts."""
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xlsm", ".xls"]:
        try:
            return pd.read_excel(path, engine="openpyxl")
        except ImportError:
            from openpyxl import load_workbook
            wb = load_workbook(path, read_only=True, data_only=True)
            ws = wb.active
            rows = ws.iter_rows(values_only=True)
            header = list(next(rows))
            data = list(rows)
            wb.close()
            return pd.DataFrame(data, columns=header)
    raise ValueError(f"Unsupported format: {path}")


def normalize_sensor_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize sensor-column names."""
    mapping = {}
    for col in df.columns:
        m = re.match(r"^Sensors_\s*(\d+)$", str(col))
        if m:
            mapping[col] = f"Sensors_{int(m.group(1)):03d}"
    return df.rename(columns=mapping)


def get_sensor_cols(df: pd.DataFrame):
    cols = [c for c in df.columns if re.match(r"^Sensors_\d{3}$", str(c))]
    return sorted(cols, key=lambda c: int(c.split("_")[1]))


def ensure_datetime(df, col):
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def xel_from_fem(df):
    """Project Accuracy_1 to XEL using cos(Elevation)."""
    out = df.copy()
    out["XEL"] = out["Accuracy_1"] * np.cos(np.deg2rad(out["Elevation"]))
    return out


In [ ]:
real_path = find_data_file(REAL_CANDIDATES)
print('Real dataset:', real_path)

real_raw = read_table(real_path)
real_raw = normalize_sensor_columns(real_raw)
real_raw = ensure_datetime(real_raw, "Timestamp")
real_raw = real_raw.sort_values("Timestamp").reset_index(drop=True)

sensor_cols = get_sensor_cols(real_raw)
print(f"Real rows: {len(real_raw):,} | Sensors: {len(sensor_cols)}")
print('Time range:', real_raw["Timestamp"].min(), "->", real_raw["Timestamp"].max())
print('Valid XEL targets:', real_raw["XEl Offset"].notna().sum(), "| EL:", real_raw["El Offset"].notna().sum())
print('Valid elevations:', real_raw["SRT Elevation Ange[DEG]"].notna().sum())
if "Azimuth" in real_raw.columns:
    print('Azimuth available:', real_raw["Azimuth"].notna().sum(), "| range:", real_raw["Azimuth"].min(), "->", real_raw["Azimuth"].max())
else:
    print('Azimuth is not present in the real dataset.')

real_raw[["SRT Elevation Ange[DEG]", "El Offset", "XEl Offset"]].describe()


In [ ]:
TRUE_XEL_AUDIT_CANDIDATES = ["control_crosscheck_effective_target_projection.csv"]
true_xel_path = find_data_file(TRUE_XEL_AUDIT_CANDIDATES)
print('XEL target audit:', true_xel_path)

true_xel_audit = pd.read_csv(true_xel_path)
true_xel_audit["Timestamp"] = pd.to_datetime(
    true_xel_audit["Timestamp"], format="%Y%m%d %H:%M:%S.%f", errors="coerce"
)
true_xel_audit = true_xel_audit.dropna(subset=["Timestamp", "projected_true_XEL"]).copy()
true_xel_audit = true_xel_audit.sort_values("Timestamp").drop_duplicates("Timestamp", keep="last")

target_mask_before = real_raw["XEl Offset"].notna()
real_raw["XEl Offset_longitudinal_original"] = real_raw["XEl Offset"]
audit_by_time = true_xel_audit.set_index("Timestamp")

mapped_true_xel = real_raw["Timestamp"].map(audit_by_time["projected_true_XEL"])
matched_targets = target_mask_before & mapped_true_xel.notna()
if int(matched_targets.sum()) != int(target_mask_before.sum()):
    missing = real_raw.loc[target_mask_before & mapped_true_xel.isna(), "Timestamp"]
    raise ValueError(f"Targets without an XEL audit: {len(missing)}. Examples: {missing.head().tolist()}")

real_raw.loc[target_mask_before, "XEl Offset"] = mapped_true_xel[target_mask_before]
for source_col, dest_col in [
    ("receiver_class", "ReceiverCode"),
    ("receiver_band", "ReceiverBand"),
    ("azel_nonzero_count_since_zero", "ActiveAzelOffsetCount"),
    ("fit_xel_sigma_arcsec", "FitXelSigmaArcsec"),
    ("fit_el_sigma_arcsec", "FitElSigmaArcsec"),
    ("lonfit_hpbw_arcsec", "LonFitHPBWArcsec"),
    ("latfit_hpbw_arcsec", "LatFitHPBWArcsec"),
]:
    if source_col in audit_by_time.columns:
        real_raw[dest_col] = real_raw["Timestamp"].map(audit_by_time[source_col])

active_offset_mask = target_mask_before & (pd.to_numeric(real_raw["ActiveAzelOffsetCount"], errors="coerce").fillna(0) > 0)
active_offset_targets_df = real_raw.loc[active_offset_mask, [
    "Timestamp", "SRT Elevation Ange[DEG]", "Azimuth",
    "XEl Offset_longitudinal_original", "XEl Offset", "ActiveAzelOffsetCount",
    "ReceiverCode", "ReceiverBand",
]].copy()
print('Original targets mapped:', int(matched_targets.sum()))
print('Samples excluded for active azelOffsets:', int(active_offset_mask.sum()))
display(active_offset_targets_df)


real_raw.loc[active_offset_mask, "XEl Offset"] = np.nan

projection_summary_df = pd.DataFrame([
    {
        "target": "longitudinal_stored",
        "count": int(target_mask_before.sum()),
        "median_abs": float(real_raw.loc[target_mask_before, "XEl Offset_longitudinal_original"].abs().median()),
        "p90_abs": float(real_raw.loc[target_mask_before, "XEl Offset_longitudinal_original"].abs().quantile(.90)),
        "gt15_percent": float((real_raw.loc[target_mask_before, "XEl Offset_longitudinal_original"].abs() > 15).mean() * 100),
    },
    {
        "target": "true_XEL_before_procedural_exclusion",
        "count": int(mapped_true_xel[target_mask_before].notna().sum()),
        "median_abs": float(mapped_true_xel[target_mask_before].abs().median()),
        "p90_abs": float(mapped_true_xel[target_mask_before].abs().quantile(.90)),
        "gt15_percent": float((mapped_true_xel[target_mask_before].abs() > 15).mean() * 100),
    },
])
display(projection_summary_df)


In [ ]:
def audit_real_dataset(df, sensor_cols):
    print('Missing sensors:', int(df[sensor_cols].isna().sum().sum()))
    dt = df["Timestamp"].diff().dt.total_seconds().dropna()
    print('Temperature sampling interval [s]')
    print(dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    target_df = df[df[TARGET].notna()].copy()
    target_dt = target_df["Timestamp"].diff().dt.total_seconds().dropna()
    print('\nValid cross-scan targets:', len(target_df))
    if len(target_dt):
        print('Time between targets [s]')
        print(target_dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    flat = df[sensor_cols].to_numpy(dtype="float32").ravel()
    flat = flat[np.isfinite(flat)]
    print('\nReal temperatures: mean/std/min/max', np.mean(flat), np.std(flat), np.min(flat), np.max(flat))

audit_real_dataset(real_raw, sensor_cols)


In [ ]:
def _impute_sensor_frame(sensor_df, impute_limit=3):
    # Limited backward filling is offline-only and needs an online availability audit.
    return sensor_df.astype("float32").ffill(limit=impute_limit).bfill(limit=1)


def build_supervised_windows_raw(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    min_step_sec=MIN_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build consecutive-row windows, rejecting duplicate timestamps."""
    df = df.sort_values("Timestamp").reset_index(drop=True).copy()
    df[sensor_cols] = _impute_sensor_frame(df[sensor_cols], impute_limit=impute_limit)

    if target_table is not None:
        print('Azimuth in target_table is fully supported by resampled_grid; raw_rows uses targets from the thermal dataset.')
    target_mask = df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()
    target_positions = np.flatnonzero(target_mask.to_numpy())

    X, elev, y, times = [], [], [], []
    skip = {"too_early": 0, "time_gap": 0, "too_close": 0, "nan": 0, "thermal_jump": 0}

    for pos in target_positions:
        end = pos + 1 if include_current else pos
        start = end - seq_len
        if start < 0:
            skip["too_early"] += 1
            continue

        window = df.iloc[start:end]
        dts = window["Timestamp"].diff().dt.total_seconds().dropna()
        if len(dts) != seq_len - 1 or (dts > max_gap_sec).any():
            skip["time_gap"] += 1
            continue
        if (dts < min_step_sec).any():
            skip["too_close"] += 1
            continue

        values = window[sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(df.loc[pos, "SRT Elevation Ange[DEG]"]))
        y.append(float(df.loc[pos, target_col]))
        times.append(df.loc[pos, "Timestamp"])

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Target": y})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows_resampled(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    expected_step_sec=EXPECTED_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build two-minute windows, aggregating duplicate rows and checking gaps and thermal jumps."""
    df = df.sort_values("Timestamp").copy()
    if target_table is not None:
        target_df = target_table.rename(columns={"Elevation": "SRT Elevation Ange[DEG]"}).copy()
    else:
        target_df = df[df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()].copy()


    sensor_ts = df[["Timestamp"] + sensor_cols].dropna(subset=["Timestamp"]).copy()
    sensor_ts["GridTime"] = sensor_ts["Timestamp"].dt.floor(f"{expected_step_sec}s")
    grid = sensor_ts.groupby("GridTime", sort=True)[sensor_cols].mean()
    grid = _impute_sensor_frame(grid, impute_limit=impute_limit)

    X, elev, azim, y, times, filenames = [], [], [], [], [], []
    skip = {"too_early": 0, "missing_grid": 0, "nan": 0, "thermal_jump": 0}
    step = pd.Timedelta(seconds=expected_step_sec)

    for _, row in target_df.iterrows():
        target_time = row["Timestamp"]
        end_time = target_time.floor(f"{expected_step_sec}s") if include_current else (target_time.floor(f"{expected_step_sec}s") - step)
        expected_index = pd.date_range(end=end_time, periods=seq_len, freq=step)

        if len(grid) == 0 or expected_index[0] < grid.index.min():
            skip["too_early"] += 1
            continue

        # Reject incomplete grids rather than filling long resampling gaps.
        if not expected_index.isin(grid.index).all():
            skip["missing_grid"] += 1
            continue

        values = grid.loc[expected_index, sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(row["SRT Elevation Ange[DEG]"]))
        azim.append(float(row["Azimuth"]) if "Azimuth" in row and pd.notna(row["Azimuth"]) else np.nan)
        y.append(float(row[target_col]))
        times.append(target_time)
        filenames.append(row["Filename"] if "Filename" in row else None)

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Azimuth": azim, "Target": y, "Filename": filenames})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows(df, sensor_cols, **kwargs):
    if USE_RESAMPLED_WINDOWS:
        return build_supervised_windows_resampled(df, sensor_cols, **kwargs)
    return build_supervised_windows_raw(df, sensor_cols, **kwargs)

target_source = pointing_targets if USE_POINTING_TARGETS_WITH_AZIMUTH and "pointing_targets" in globals() and pointing_targets is not None else None
X_seq, X_elev, y, meta, skip = build_supervised_windows(real_raw, sensor_cols, target_table=target_source)
print('Window mode:', "resampled_grid" if USE_RESAMPLED_WINDOWS else "raw_rows")
print('Target source:', "pointing_offsets_real_azimuth" if target_source is not None else 'thermal_dataset')
print('Windows with azimuth:', int(meta["Azimuth"].notna().sum()) if "Azimuth" in meta else 0)
print('Valid supervised samples:', len(y))
print('Rejected by timing/jump checks:', skip)
print("Shape X_seq:", X_seq.shape, "X_elev:", X_elev.shape, "y:", y.shape)
meta.describe(include="all")


In [ ]:
audit_meta_cols = [
    "Timestamp", "receiver_class", "receiver_band", "fit_xel_sigma_arcsec",
    "fit_el_sigma_arcsec", "lonfit_hpbw_arcsec", "latfit_hpbw_arcsec",
    "azel_nonzero_count_since_zero",
]
available_audit_cols = [c for c in audit_meta_cols if c in true_xel_audit.columns]
meta = meta.merge(true_xel_audit[available_audit_cols], on="Timestamp", how="left", validate="one_to_one")
meta = meta.rename(columns={
    "receiver_class": "ReceiverCode",
    "receiver_band": "ReceiverBand",
    "fit_xel_sigma_arcsec": "FitXelSigmaArcsec",
    "fit_el_sigma_arcsec": "FitElSigmaArcsec",
    "lonfit_hpbw_arcsec": "LonFitHPBWArcsec",
    "latfit_hpbw_arcsec": "LatFitHPBWArcsec",
    "azel_nonzero_count_since_zero": "ActiveAzelOffsetCount",
})
print('Windows with missing receiver metadata:', int(meta["ReceiverBand"].isna().sum()))
display(meta["ReceiverBand"].value_counts(dropna=False).rename_axis("receiver").reset_index(name="count"))


In [ ]:
def window_quality_report(X, meta):
    if len(X) == 0:
        print('No valid windows.')
        return pd.DataFrame()
    diffs = np.diff(X, axis=1)
    q = pd.DataFrame({
        "Timestamp": meta["Timestamp"].to_numpy(),
        "Elevation": meta["Elevation"].to_numpy(),
        "Target": meta["Target"].to_numpy(),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "current_mean": X[:, -1, :].mean(axis=1),
        "trend_mean": (X[:, -1, :] - X[:, 0, :]).mean(axis=1),
        "abs_rate_mean": np.abs(diffs).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diffs).max(axis=(1, 2)),
    })
    print(q[["window_mean", "window_std", "current_mean", "trend_mean", "abs_rate_mean", "abs_rate_max"]].describe().to_string())
    suspect = q[(q["abs_rate_max"] > MAX_SENSOR_RATE_C) | (q["abs_rate_mean"] > 0.5)]
    print('Remaining suspect windows:', len(suspect))
    if len(suspect):
        display(suspect.sort_values("abs_rate_max", ascending=False).head(20))
    return q

quality_df = window_quality_report(X_seq, meta)


In [ ]:
def chronological_split(meta, train_frac=0.60, val_frac=0.20):
    order = np.argsort(meta["Timestamp"].to_numpy())
    n = len(order)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_idx = order[:n_train]
    val_idx = order[n_train:n_train + n_val]
    test_idx = order[n_train + n_val:]
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = chronological_split(meta)
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))
print("Train:", meta.iloc[train_idx]["Timestamp"].min(), "->", meta.iloc[train_idx]["Timestamp"].max())
print("Val  :", meta.iloc[val_idx]["Timestamp"].min(), "->", meta.iloc[val_idx]["Timestamp"].max())
print("Test :", meta.iloc[test_idx]["Timestamp"].min(), "->", meta.iloc[test_idx]["Timestamp"].max())

plt.figure(figsize=(10, 4))
plt.plot(meta["Timestamp"].to_numpy(), meta["Target"].to_numpy(), ".", alpha=0.75)
plt.axvspan(meta.iloc[train_idx]["Timestamp"].min(), meta.iloc[train_idx]["Timestamp"].max(), alpha=0.08, label="train")
plt.axvspan(meta.iloc[val_idx]["Timestamp"].min(), meta.iloc[val_idx]["Timestamp"].max(), alpha=0.08, label="val")
plt.axvspan(meta.iloc[test_idx]["Timestamp"].min(), meta.iloc[test_idx]["Timestamp"].max(), alpha=0.08, label="test")
plt.title(f"Observed target {TARGET} and chronological split")
plt.ylabel("arcsec")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
def scalar_physical_features(X, elev):
    current = X[:, -1, :]
    start = X[:, 0, :]
    mean_time = X.mean(axis=1)
    std_time = X.std(axis=1)
    diff = np.diff(X, axis=1)
    trend = current - start

    feats = pd.DataFrame({
        "elevation": elev.ravel(),
        "sin_el": np.sin(np.deg2rad(elev.ravel())),
        "cos_el": np.cos(np.deg2rad(elev.ravel())),
        "current_mean": current.mean(axis=1),
        "current_std_spatial": current.std(axis=1),
        "current_min": current.min(axis=1),
        "current_max": current.max(axis=1),
        "current_range": current.max(axis=1) - current.min(axis=1),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "mean_sensor_std_time": std_time.mean(axis=1),
        "trend_mean": trend.mean(axis=1),
        "trend_std": trend.std(axis=1),
        "trend_range": trend.max(axis=1) - trend.min(axis=1),
        "abs_rate_mean": np.abs(diff).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diff).max(axis=(1, 2)),
    })
    return feats

def temporal_context_features(meta_df):
    """Build cyclic time features as seasonal and diurnal proxies."""
    ts = pd.to_datetime(meta_df["Timestamp"])
    hour = ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0
    doy = ts.dt.dayofyear.astype(float)
    out = pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour / 24.0),
        "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25),
        "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        "month": ts.dt.month.astype(float),
    }).reset_index(drop=True)
    if "Azimuth" in meta_df.columns and meta_df["Azimuth"].notna().any():
        az = pd.to_numeric(meta_df["Azimuth"], errors="coerce").fillna(meta_df["Azimuth"].median())
        out["az_sin"] = np.sin(np.deg2rad(az))
        out["az_cos"] = np.cos(np.deg2rad(az))
    return out

class FeatureBuilder:
    """Fit scalar features and current-state/trend PCA on training data only."""
    def __init__(self, n_pca_current=8, n_pca_trend=8):
        self.n_pca_current = n_pca_current
        self.n_pca_trend = n_pca_trend
        self.scaler_current = StandardScaler()
        self.scaler_trend = StandardScaler()
        self.pca_current = PCA(n_components=n_pca_current, random_state=RANDOM_SEED)
        self.pca_trend = PCA(n_components=n_pca_trend, random_state=RANDOM_SEED)

    def fit(self, X, elev):
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        self.pca_current.fit(self.scaler_current.fit_transform(current))
        self.pca_trend.fit(self.scaler_trend.fit_transform(trend))
        return self

    def transform(self, X, elev, meta_df=None):
        base = scalar_physical_features(X, elev).reset_index(drop=True)
        if meta_df is not None:
            base = pd.concat([base, temporal_context_features(meta_df)], axis=1)
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        pc_cur = self.pca_current.transform(self.scaler_current.transform(current))
        pc_tr = self.pca_trend.transform(self.scaler_trend.transform(trend))
        for i in range(pc_cur.shape[1]):
            base[f"pc_current_{i+1:02d}"] = pc_cur[:, i]
        for i in range(pc_tr.shape[1]):
            base[f"pc_trend_{i+1:02d}"] = pc_tr[:, i]
        return base

fb = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq[train_idx], X_elev[train_idx])
F_train = fb.transform(X_seq[train_idx], X_elev[train_idx], meta.iloc[train_idx])
F_val = fb.transform(X_seq[val_idx], X_elev[val_idx], meta.iloc[val_idx])
F_test = fb.transform(X_seq[test_idx], X_elev[test_idx], meta.iloc[test_idx])

y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(F_train.shape)
F_train.head()


nan_report = pd.DataFrame({
    "train_nan": F_train.isna().sum(),
    "val_nan": F_val.isna().sum(),
    "test_nan": F_test.isna().sum(),
})
nan_report = nan_report[(nan_report["train_nan"] > 0) | (nan_report["val_nan"] > 0) | (nan_report["test_nan"] > 0)]
if len(nan_report):
    print('Features containing NaN before imputation:')
    display(nan_report.sort_values(["val_nan", "test_nan", "train_nan"], ascending=False))
else:
    print('No NaN values in tabular features.')


In [ ]:
def eval_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

models = {
    "median_baseline": DummyRegressor(strategy="median"),
    "ridge_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))
    ]),
    "huber_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=2000))
    ]),
    "elasticnet_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.8, 0.95], alphas=np.logspace(-3, 2, 30), cv=5, random_state=RANDOM_SEED, max_iter=20000))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1
        )),
    ]),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.03, l2_regularization=0.1, min_samples_leaf=12, random_state=RANDOM_SEED
    ),
}

results = []
val_predictions = {}
test_predictions = {}

for name, model in models.items():
    m = clone(model)
    m.fit(F_train, y_train)
    pred_val = m.predict(F_val)
    pred_test = m.predict(F_test)
    val_predictions[name] = pred_val
    test_predictions[name] = pred_test
    row = eval_predictions(name + " / val", y_val, pred_val)
    row_test = eval_predictions(name + " / test", y_test, pred_test)
    results.extend([row, row_test])

results_df = pd.DataFrame(results).sort_values(["MAE"])
results_df


In [ ]:
old_target_by_time = true_xel_audit.set_index("Timestamp")["stored_XEl_Offset"]
y_old_longitudinal = meta["Timestamp"].map(old_target_by_time).to_numpy(dtype=float)
target_ablation_rows = []
for target_name, target_values in [
    ("old_longitudinal", y_old_longitudinal),
    ("true_XEL", y.astype(float)),
]:
    for model_name, estimator in {
        "median": models["median_baseline"],
        "ridge": models["ridge_physical"],
        "huber": models["huber_physical"],
        "random_forest": models["random_forest"],
    }.items():
        fitted = clone(estimator).fit(F_train, target_values[train_idx])
        for split_name, idx, Xuse in [("val", val_idx, F_val), ("test", test_idx, F_test)]:
            pred = fitted.predict(Xuse)
            err = np.abs(target_values[idx] - pred)
            target_ablation_rows.append({
                "target": target_name, "model": model_name, "split": split_name,
                "count": len(idx), "MAE": float(err.mean()),
                "within_5_%": float((err <= SPEC_ARCSEC).mean()*100),
                "p90_abs_error": float(np.quantile(err, .90)),
            })
target_definition_ablation_df = pd.DataFrame(target_ablation_rows).sort_values(["split", "target", "MAE"])
display(target_definition_ablation_df)


In [ ]:
RECEIVER_LEVELS = ["K", "C_high", "C_low", "Q", "unknown"]

def receiver_one_hot(meta_df):
    band = meta_df["ReceiverBand"].fillna("unknown").astype(str)
    cat = pd.Categorical(band, categories=RECEIVER_LEVELS)
    return pd.get_dummies(cat, prefix="rx", dtype=float).reset_index(drop=True)

R_all = receiver_one_hot(meta)
R_train = R_all.iloc[train_idx].reset_index(drop=True)
R_val = R_all.iloc[val_idx].reset_index(drop=True)
R_test = R_all.iloc[test_idx].reset_index(drop=True)

F_train_rx = pd.concat([F_train.reset_index(drop=True), R_train], axis=1)
F_val_rx = pd.concat([F_val.reset_index(drop=True), R_val], axis=1)
F_test_rx = pd.concat([F_test.reset_index(drop=True), R_test], axis=1)

ablation_models = {
    "ridge": models["ridge_physical"],
    "huber": models["huber_physical"],
    "random_forest": models["random_forest"],
}
receiver_ablation_rows = []
receiver_predictions = {}
for feature_set, Xtr, Xv, Xte in [
    ("without_receiver", F_train, F_val, F_test),
    ("with_receiver", F_train_rx, F_val_rx, F_test_rx),
]:
    for model_name, estimator in ablation_models.items():
        fitted = clone(estimator).fit(Xtr, y_train)
        pv = fitted.predict(Xv)
        pt = fitted.predict(Xte)
        receiver_predictions[(feature_set, model_name, "val")] = pv
        receiver_predictions[(feature_set, model_name, "test")] = pt
        for split_name, yt, pred in [("val", y_val, pv), ("test", y_test, pt)]:
            row = eval_predictions(f"{feature_set}:{model_name} / {split_name}", yt, pred)
            row.update({
                "feature_set": feature_set,
                "base_model": model_name,
                "split": split_name,
                "within_5_%": float((np.abs(yt - pred) <= SPEC_ARCSEC).mean() * 100),
                "p90_abs_error": float(np.quantile(np.abs(yt - pred), .90)),
            })
            receiver_ablation_rows.append(row)

receiver_ablation_df = pd.DataFrame(receiver_ablation_rows).sort_values(["split", "MAE"])
display(receiver_ablation_df)


pretest_idx = np.sort(np.concatenate([train_idx, val_idx]))
R_cv_source = receiver_one_hot(meta)
tscv_rx = TimeSeriesSplit(n_splits=4)
cv_rows = []
for fold, (tr_local, ev_local) in enumerate(tscv_rx.split(pretest_idx), 1):
    tr = pretest_idx[tr_local]
    ev = pretest_idx[ev_local]

    fb_fold = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq[tr], X_elev[tr])
    Xtr_base = fb_fold.transform(X_seq[tr], X_elev[tr], meta.iloc[tr]).reset_index(drop=True)
    Xev_base = fb_fold.transform(X_seq[ev], X_elev[ev], meta.iloc[ev]).reset_index(drop=True)
    for feature_set in ["without_receiver", "with_receiver"]:
        if feature_set == "with_receiver":
            Xtr_use = pd.concat([Xtr_base, R_cv_source.iloc[tr].reset_index(drop=True)], axis=1)
            Xev_use = pd.concat([Xev_base, R_cv_source.iloc[ev].reset_index(drop=True)], axis=1)
        else:
            Xtr_use, Xev_use = Xtr_base, Xev_base
        for model_name, estimator in ablation_models.items():
            fitted = clone(estimator).fit(Xtr_use, y[tr])
            pred = fitted.predict(Xev_use)
            cv_rows.append({
                "fold": fold, "feature_set": feature_set, "model": model_name,
                "n_train": len(tr), "n_eval": len(ev),
                "MAE": mean_absolute_error(y[ev], pred),
                "within_5_%": float((np.abs(y[ev] - pred) <= SPEC_ARCSEC).mean() * 100),
            })
receiver_ablation_cv_df = pd.DataFrame(cv_rows)
receiver_ablation_cv_summary_df = receiver_ablation_cv_df.groupby(["feature_set", "model"]).agg(
    folds=("fold", "nunique"), MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    within_5_mean=("within_5_%", "mean"),
).reset_index().sort_values("MAE_mean")
display(receiver_ablation_cv_summary_df)


best_receiver_row = receiver_ablation_df[receiver_ablation_df["split"] == "val"].sort_values("MAE").iloc[0]
BEST_RX_FEATURE_SET = best_receiver_row["feature_set"]
BEST_RX_MODEL = best_receiver_row["base_model"]
best_rx_test_pred = receiver_predictions[(BEST_RX_FEATURE_SET, BEST_RX_MODEL, "test")]
print('Validation selection:', BEST_RX_FEATURE_SET, BEST_RX_MODEL)


In [ ]:
best_name = results_df[results_df["model"].str.endswith("/ val")].iloc[0]["model"].replace(" / val", "")
print('Best model on validation:', best_name)

pred_test = test_predictions[best_name]
resid = y_test - pred_test

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].plot(meta.iloc[test_idx]["Timestamp"].to_numpy(), y_test, label='observed', marker="o")
axs[0].plot(meta.iloc[test_idx]["Timestamp"].to_numpy(), pred_test, label='predicted', marker="x")
axs[0].set_title(f"Chronological test - {best_name}")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].scatter(y_test, pred_test, alpha=0.8)
lo = min(y_test.min(), pred_test.min())
hi = max(y_test.max(), pred_test.max())
axs[1].plot([lo, hi], [lo, hi], "k--")
axs[1].set_xlabel('observed [arcsec]')
axs[1].set_ylabel('predicted [arcsec]')
axs[1].set_title('Predicted versus observed')
axs[1].grid(True, alpha=0.3)

axs[2].hist(resid, bins=20, alpha=0.75)
axs[2].axvline(0, color="k", linestyle="--")
axs[2].set_title(f"Residuals | MAE={mean_absolute_error(y_test, pred_test):.2f} arcsec")
axs[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def local_repeatability_check(meta, max_dt_hours=6, max_delev_deg=3.0):
    rows = []
    m = meta.sort_values("Timestamp").reset_index(drop=True)
    for i in range(len(m)):
        t0 = m.loc[i, "Timestamp"]
        e0 = m.loc[i, "Elevation"]
        y0 = m.loc[i, "Target"]
        later = m.iloc[i+1:]
        if later.empty:
            continue
        dt_h = (later["Timestamp"] - t0).dt.total_seconds() / 3600
        cand = later[(dt_h > 0) & (dt_h <= max_dt_hours) & ((later["Elevation"] - e0).abs() <= max_delev_deg)]
        for _, r in cand.iterrows():
            rows.append({
                "dt_h": (r["Timestamp"] - t0).total_seconds() / 3600,
                "delev": abs(r["Elevation"] - e0),
                "abs_dtarget": abs(r["Target"] - y0),
            })
    return pd.DataFrame(rows)

repeat_df = local_repeatability_check(meta)
print('Nearby pairs found:', len(repeat_df))
if len(repeat_df):
    display(repeat_df.describe(percentiles=[0.5, 0.75, 0.9, 0.95]))
    plt.figure(figsize=(6,4))
    plt.scatter(repeat_df["dt_h"], repeat_df["abs_dtarget"], alpha=0.6)
    plt.xlabel('Time difference [h]')
    plt.ylabel("|Delta target| [arcsec]")
    plt.title('Local repeatability of the real target')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Few nearby scans; consecutive calibrations are needed to estimate the noise floor reliably.')


In [ ]:
from scipy.stats import norm

def circular_distance_deg(a, b):
    return np.abs((np.asarray(a, dtype=float) - np.asarray(b, dtype=float) + 180.0) % 360.0 - 180.0)

def bootstrap_quantile_ci(values, quantile=.5, n_boot=3000, random_state=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(random_state)
    stats = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        stats[b] = np.quantile(rng.choice(values, size=len(values), replace=True), quantile)
    return tuple(np.quantile(stats, [.025, .975]))

def build_repeatability_pairs(meta_df, sequences, max_dt_hours=6.0, max_delev_deg=3.0, fit_sigma_col="FitXelSigmaArcsec"):
    ordered = meta_df.copy().reset_index().rename(columns={"index": "sample_idx"})
    ordered = ordered.sort_values("Timestamp").reset_index(drop=True)
    current = np.asarray(sequences[:, -1, :], dtype=float)
    rows = []
    for i in range(len(ordered) - 1):
        left = ordered.iloc[i]
        later = ordered.iloc[i + 1:]
        dt_h = (later["Timestamp"] - left["Timestamp"]).dt.total_seconds() / 3600.0
        candidates = later[
            (dt_h > 0) & (dt_h <= max_dt_hours) &
            ((later["Elevation"] - left["Elevation"]).abs() <= max_delev_deg)
        ]
        a = current[int(left["sample_idx"])]
        a_centered = a - np.nanmean(a)
        for _, right in candidates.iterrows():
            b = current[int(right["sample_idx"])]
            b_centered = b - np.nanmean(b)
            sigma_a = pd.to_numeric(pd.Series([left.get(fit_sigma_col, np.nan)]), errors="coerce").iloc[0]
            sigma_b = pd.to_numeric(pd.Series([right.get(fit_sigma_col, np.nan)]), errors="coerce").iloc[0]
            pair_sigma = np.sqrt(sigma_a**2 + sigma_b**2) if np.isfinite(sigma_a) and np.isfinite(sigma_b) else np.nan
            rows.append({
                "left_idx": int(left["sample_idx"]), "right_idx": int(right["sample_idx"]),
                "left_time": left["Timestamp"], "right_time": right["Timestamp"],
                "dt_min": (right["Timestamp"] - left["Timestamp"]).total_seconds() / 60.0,
                "delev_deg": abs(float(right["Elevation"] - left["Elevation"])),
                "daz_deg": float(circular_distance_deg(right["Azimuth"], left["Azimuth"])),
                "same_receiver_band": left.get("ReceiverBand") == right.get("ReceiverBand"),
                "same_receiver_code": left.get("ReceiverCode") == right.get("ReceiverCode"),
                "receiver_band": left.get("ReceiverBand") if left.get("ReceiverBand") == right.get("ReceiverBand") else "mixed",
                "abs_dtarget": abs(float(right["Target"] - left["Target"])),
                "signed_dtarget": float(right["Target"] - left["Target"]),
                "thermal_mean_delta_C": abs(float(np.nanmean(b) - np.nanmean(a))),
                "thermal_centered_rmse_C": float(np.sqrt(np.nanmean((b_centered - a_centered) ** 2))),
                "fit_pair_sigma_arcsec": pair_sigma,
            })
    return pd.DataFrame(rows)

repeatability_pairs_df = build_repeatability_pairs(meta, X_seq)
repeatability_criteria = [
    ("broad_6h_el3", 360, 3.0, None, False, None, None),
    ("same_rx_60m_el1", 60, 1.0, None, True, None, None),
    ("same_rx_60m_el1_az10", 60, 1.0, 10.0, True, None, None),
    ("same_rx_30m_el0.5_az5", 30, .5, 5.0, True, None, None),
    ("same_rx_60m_el1_thermal", 60, 1.0, None, True, .5, .5),
    ("strict_same_rx_geometry_thermal", 30, .5, 5.0, True, .5, .5),
]

repeatability_rows = []
for name, max_dt, max_el, max_az, same_rx, max_mean_t, max_pattern_t in repeatability_criteria:
    mask = (repeatability_pairs_df["dt_min"] <= max_dt) & (repeatability_pairs_df["delev_deg"] <= max_el)
    if max_az is not None:
        mask &= repeatability_pairs_df["daz_deg"] <= max_az
    if same_rx:
        mask &= repeatability_pairs_df["same_receiver_code"]
    if max_mean_t is not None:
        mask &= repeatability_pairs_df["thermal_mean_delta_C"] <= max_mean_t
    if max_pattern_t is not None:
        mask &= repeatability_pairs_df["thermal_centered_rmse_C"] <= max_pattern_t
    sub = repeatability_pairs_df.loc[mask]
    med = float(sub["abs_dtarget"].median()) if len(sub) else np.nan
    ci_low, ci_high = bootstrap_quantile_ci(sub["abs_dtarget"], random_state=RANDOM_SEED + len(repeatability_rows))
    # The iid Gaussian equivalent is diagnostic, not a physical noise floor.
    gaussian_single_mae = med / (norm.ppf(.75) * np.sqrt(np.pi)) if np.isfinite(med) else np.nan
    repeatability_rows.append({
        "criterion": name, "pair_count": len(sub),
        "median_abs_delta_target": med, "median_ci95_low": ci_low, "median_ci95_high": ci_high,
        "p75_abs_delta_target": float(sub["abs_dtarget"].quantile(.75)) if len(sub) else np.nan,
        "p90_abs_delta_target": float(sub["abs_dtarget"].quantile(.90)) if len(sub) else np.nan,
        "gaussian_equiv_single_measurement_MAE": gaussian_single_mae,
        "median_fit_pair_sigma": float(sub["fit_pair_sigma_arcsec"].median()) if len(sub) else np.nan,
        "median_thermal_mean_delta_C": float(sub["thermal_mean_delta_C"].median()) if len(sub) else np.nan,
        "median_thermal_pattern_rmse_C": float(sub["thermal_centered_rmse_C"].median()) if len(sub) else np.nan,
    })
repeatability_conditioned_df = pd.DataFrame(repeatability_rows)
display(repeatability_conditioned_df)

strict_mask = (
    (repeatability_pairs_df["dt_min"] <= 60) &
    (repeatability_pairs_df["delev_deg"] <= 1.0) &
    repeatability_pairs_df["same_receiver_code"] &
    (repeatability_pairs_df["thermal_mean_delta_C"] <= .5) &
    (repeatability_pairs_df["thermal_centered_rmse_C"] <= .5)
)
strict_pairs = repeatability_pairs_df.loc[strict_mask].copy()
repeatability_by_receiver_df = strict_pairs.groupby("receiver_band", observed=True).agg(
    pair_count=("abs_dtarget", "size"),
    median_abs_delta_target=("abs_dtarget", "median"),
    p90_abs_delta_target=("abs_dtarget", lambda s: s.quantile(.90)),
    median_fit_pair_sigma=("fit_pair_sigma_arcsec", "median"),
).reset_index()
display(repeatability_by_receiver_df)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
plot_rep = repeatability_conditioned_df[repeatability_conditioned_df["pair_count"] > 0]
axs[0].barh(plot_rep["criterion"], plot_rep["median_abs_delta_target"], color="tab:blue", alpha=.8)
axs[0].axvline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[0].set_xlabel('Median |delta XEL| [arcsec]')
axs[0].set_title('Repeatability under increasingly strict conditions')
axs[0].grid(True, axis="x", alpha=.25)
axs[1].scatter(repeatability_pairs_df["thermal_centered_rmse_C"], repeatability_pairs_df["abs_dtarget"],
               c=repeatability_pairs_df["dt_min"], cmap="viridis", alpha=.65, s=24)
axs[1].axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[1].set_xlabel('Centered thermal-pattern RMSE [C]')
axs[1].set_ylabel("|delta XEL| [arcsec]")
axs[1].set_title('Target variability versus thermal change')
axs[1].grid(True, alpha=.25)
plt.tight_layout()
plt.show()


In [ ]:
test_diagnostics = meta.iloc[test_idx].copy().reset_index(drop=True)
test_diagnostics["y_true"] = y_test
test_diagnostics["y_pred"] = best_rx_test_pred
test_diagnostics["residual"] = test_diagnostics["y_true"] - test_diagnostics["y_pred"]
test_diagnostics["abs_error"] = test_diagnostics["residual"].abs()
test_diagnostics["within_5"] = test_diagnostics["abs_error"] <= SPEC_ARCSEC
test_diagnostics["elev_band"] = pd.cut(
    test_diagnostics["Elevation"], bins=[0, 20, 30, 45, 60, 75, 80, 90], include_lowest=True
)

def diagnostic_group(df, group_col):
    return df.groupby(group_col, observed=True).agg(
        count=("abs_error", "size"),
        MAE=("abs_error", "mean"),
        within_5_percent=("within_5", lambda s: s.mean() * 100),
        p90_abs_error=("abs_error", lambda s: s.quantile(.90)),
        median_abs_target=("y_true", lambda s: s.abs().median()),
        median_HPBW_lon=("LonFitHPBWArcsec", "median"),
        median_fit_sigma=("FitXelSigmaArcsec", "median"),
    ).reset_index()

performance_by_elevation_df = diagnostic_group(test_diagnostics, "elev_band")
performance_by_receiver_df = diagnostic_group(test_diagnostics, "ReceiverBand")
display(performance_by_elevation_df)
display(performance_by_receiver_df)

receiver_fit_audit_df = meta.groupby(["ReceiverCode", "ReceiverBand"], observed=True).agg(
    count=("Target", "size"),
    median_HPBW_lon=("LonFitHPBWArcsec", "median"),
    p10_HPBW_lon=("LonFitHPBWArcsec", lambda s: s.quantile(.10)),
    p90_HPBW_lon=("LonFitHPBWArcsec", lambda s: s.quantile(.90)),
    median_HPBW_lat=("LatFitHPBWArcsec", "median"),
    median_xel_sigma=("FitXelSigmaArcsec", "median"),
    p90_xel_sigma=("FitXelSigmaArcsec", lambda s: s.quantile(.90)),
).reset_index()
display(receiver_fit_audit_df)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for band, g in test_diagnostics.groupby("ReceiverBand"):
    axs[0].scatter(g["Elevation"], g["abs_error"], label=band, alpha=.75)
axs[0].axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[0].set_xlabel('Elevation [deg]')
axs[0].set_ylabel('Absolute error [arcsec]')
axs[0].set_title('True-XEL error by elevation/receiver')
axs[0].grid(True, alpha=.25)
axs[0].legend()

plot_rx = performance_by_receiver_df.sort_values("MAE")
axs[1].bar(plot_rx["ReceiverBand"].astype(str), plot_rx["MAE"], color="tab:blue", alpha=.8)
axs[1].axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[1].set_ylabel("MAE [arcsec]")
axs[1].set_title('MAE by receiver')
axs[1].grid(True, axis="y", alpha=.25)
plt.tight_layout()
plt.show()


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
except Exception as e:
    TF_AVAILABLE = False
    print('TensorFlow is unavailable:', e)

class SequencePCA:
    def __init__(self, n_components=16):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=RANDOM_SEED)

    def fit(self, X):
        flat = X.reshape(-1, X.shape[-1])
        self.pca.fit(self.scaler.fit_transform(flat))
        return self

    def transform(self, X):
        n, t, f = X.shape
        flat = X.reshape(-1, f)
        z = self.pca.transform(self.scaler.transform(flat))
        return z.reshape(n, t, self.n_components).astype("float32")

if TF_AVAILABLE:
    seq_pca = SequencePCA(n_components=min(16, len(train_idx)-1)).fit(X_seq[train_idx])
    Z_train = seq_pca.transform(X_seq[train_idx])
    Z_val = seq_pca.transform(X_seq[val_idx])
    Z_test = seq_pca.transform(X_seq[test_idx])

    elev_scaler = StandardScaler().fit(X_elev[train_idx])
    E_train = elev_scaler.transform(X_elev[train_idx]).astype("float32")
    E_val = elev_scaler.transform(X_elev[val_idx]).astype("float32")
    E_test = elev_scaler.transform(X_elev[test_idx]).astype("float32")

    y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    ys_train = y_scaler.transform(y_train.reshape(-1, 1)).astype("float32")
    ys_val = y_scaler.transform(y_val.reshape(-1, 1)).astype("float32")

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_SEED)

    seq_in = keras.Input(shape=(SEQ_LEN, Z_train.shape[-1]), name="thermal_pca_history")
    elev_in = keras.Input(shape=(1,), name="elevation")

    x = layers.GRU(32, return_sequences=False, dropout=0.25, recurrent_dropout=0.0)(seq_in)
    x = layers.Dense(16, activation="gelu")(x)
    e = layers.Dense(8, activation="gelu")(elev_in)
    h = layers.Concatenate()([x, e])
    h = layers.Dropout(0.35)(h)
    h = layers.Dense(16, activation="gelu")(h)
    out = layers.Dense(1, dtype="float32")(h)

    gru_model = keras.Model([seq_in, elev_in], out)
    gru_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-3), loss="mae", metrics=["mae"])
    gru_model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-6),
    ]

    hist = gru_model.fit(
        [Z_train, E_train], ys_train,
        validation_data=([Z_val, E_val], ys_val),
        epochs=300,
        batch_size=16,
        callbacks=callbacks,
        verbose=0,
    )

    pred_test_s = gru_model.predict([Z_test, E_test], verbose=0)
    pred_test_gru = y_scaler.inverse_transform(pred_test_s).ravel()

    print(pd.DataFrame([
        eval_predictions("compact_pca_gru / test", y_test, pred_test_gru),
        eval_predictions(best_name + " / test", y_test, test_predictions[best_name]),
    ]))

    plt.figure(figsize=(8, 4))
    plt.plot(hist.history["loss"], label="train")
    plt.plot(hist.history["val_loss"], label="val")
    plt.title('Compact GRU - scaled loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print('Skipping GRU training.')


In [ ]:
gru_receiver_results_df = pd.DataFrame()
if TF_AVAILABLE:
    R_train_np = R_train.to_numpy(dtype="float32")
    R_val_np = R_val.to_numpy(dtype="float32")
    R_test_np = R_test.to_numpy(dtype="float32")

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_SEED)
    seq_in = keras.Input(shape=(SEQ_LEN, Z_train.shape[-1]), name="thermal_pca_history")
    elev_in = keras.Input(shape=(1,), name="elevation")
    receiver_in = keras.Input(shape=(R_train_np.shape[1],), name="receiver_one_hot")
    x = layers.GRU(32, dropout=.25)(seq_in)
    x = layers.Dense(16, activation="gelu")(x)
    e = layers.Dense(8, activation="gelu")(elev_in)
    r = layers.Dense(6, activation="gelu")(receiver_in)
    h = layers.Concatenate()([x, e, r])
    h = layers.Dropout(.35)(h)
    h = layers.Dense(16, activation="gelu")(h)
    out = layers.Dense(1, dtype="float32")(h)
    gru_receiver_model = keras.Model([seq_in, elev_in, receiver_in], out)
    gru_receiver_model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-3),
        loss="mae", metrics=["mae"],
    )
    callbacks_rx = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=.5, min_lr=1e-6),
    ]
    history_rx = gru_receiver_model.fit(
        [Z_train, E_train, R_train_np], ys_train,
        validation_data=([Z_val, E_val, R_val_np], ys_val),
        epochs=300, batch_size=16, callbacks=callbacks_rx, verbose=0,
    )
    pred_gru_rx = y_scaler.inverse_transform(
        gru_receiver_model.predict([Z_test, E_test, R_test_np], verbose=0)
    ).ravel()
    gru_receiver_results_df = pd.DataFrame([
        {**eval_predictions("compact_pca_gru_without_receiver / test", y_test, pred_test_gru),
         "within_5_%": float((np.abs(y_test-pred_test_gru) <= SPEC_ARCSEC).mean()*100)},
        {**eval_predictions("compact_pca_gru_with_receiver / test", y_test, pred_gru_rx),
         "within_5_%": float((np.abs(y_test-pred_gru_rx) <= SPEC_ARCSEC).mean()*100)},
    ])
    display(gru_receiver_results_df)
else:
    print('TensorFlow is unavailable; skipping the receiver GRU.')


In [ ]:
# This block diagnoses domain shift; it does not train the final model.

def load_fem_sample(max_rows=50000):
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        print('FEM dataset not found.')
        return None
    print("Dataset FEM:", fem_path)
    if fem_path.suffix.lower() == ".csv":
        fem = pd.read_csv(fem_path, nrows=max_rows)
    else:

        try:
            fem = pd.read_excel(fem_path, engine="openpyxl", nrows=max_rows)
        except Exception:
            from openpyxl import load_workbook
            wb = load_workbook(fem_path, read_only=True, data_only=True)
            ws = wb.active
            rows = ws.iter_rows(values_only=True)
            header = list(next(rows))
            data = []
            for i, row in enumerate(rows):
                if i >= max_rows:
                    break
                data.append(row)
            wb.close()
            fem = pd.DataFrame(data, columns=header)
    fem = normalize_sensor_columns(fem)
    return fem

fem_sample = load_fem_sample(max_rows=20000)
if fem_sample is not None:
    fem_sensor_cols = get_sensor_cols(fem_sample)
    real_flat = real_raw[sensor_cols].to_numpy(dtype="float32").ravel()
    fem_flat = fem_sample[fem_sensor_cols].to_numpy(dtype="float32").ravel()
    real_flat = real_flat[np.isfinite(real_flat)]
    fem_flat = fem_flat[np.isfinite(fem_flat)]
    print("Real temp mean/std:", real_flat.mean(), real_flat.std())
    print("FEM  temp mean/std:", fem_flat.mean(), fem_flat.std())

    plt.figure(figsize=(8, 4))
    plt.hist(fem_flat[np.random.choice(len(fem_flat), min(100000, len(fem_flat)), replace=False)], bins=80, alpha=0.5, density=True, label="FEM")
    plt.hist(real_flat[np.random.choice(len(real_flat), min(100000, len(real_flat)), replace=False)], bins=80, alpha=0.5, density=True, label='Real')
    plt.xlabel('Temperature [C]')
    plt.ylabel("densita'")
    plt.title('Temperature covariate shift')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
COMPARISON_CANDIDATES = ["ComparazioneOffsets_REALvsFEM.csv"]
comparison_path = find_data_file(COMPARISON_CANDIDATES)
cmp = pd.read_csv(comparison_path, header=[0, 1], decimal=",")
cmp_num = cmp.apply(pd.to_numeric, errors="coerce").dropna(how="all")
cmp_elev = cmp_num.iloc[:, 0].to_numpy(dtype=float)
cmp_real_long_sign = cmp_num.iloc[:, 4].to_numpy(dtype=float)
cmp_fem_xel = cmp_num.iloc[:, 5].to_numpy(dtype=float)
cmp_real_projected_file_sign = cmp_real_long_sign * np.cos(np.deg2rad(cmp_elev))


cmp_real_current_sign = -cmp_real_projected_file_sign
sign_rows = []
for convention, values in [
    ("current_true_xel_sign", cmp_real_current_sign),
    ("opposite_sign", -cmp_real_current_sign),
]:
    mask = np.isfinite(values) & np.isfinite(cmp_fem_xel)
    sign_rows.append({
        "convention": convention,
        "count": int(mask.sum()),
        "pearson": float(np.corrcoef(values[mask], cmp_fem_xel[mask])[0, 1]),
        "MAE_raw_FEM": float(mean_absolute_error(values[mask], cmp_fem_xel[mask])),
    })
fem_sign_convention_df = pd.DataFrame(sign_rows).sort_values("MAE_raw_FEM")
display(fem_sign_convention_df)
print('Notebook convention: current_true_xel_sign (#fivpt#xoffset).')

# Without timestamps, verify row order against all 868 longitudinal targets.
audit_ordered = true_xel_audit.sort_values("Timestamp").reset_index(drop=True)
if len(cmp_num) != len(audit_ordered):
    raise ValueError(f"Comparison rows={len(cmp_num)} differ from audit rows={len(audit_ordered)}")
alignment_error = np.max(np.abs(cmp_real_long_sign + audit_ordered["stored_XEl_Offset"].to_numpy(dtype=float)))
print('Maximum REAL/FEM ordering-alignment error [arcsec]:', alignment_error)
if alignment_error > 1e-3:
    raise ValueError('Comparison-row ordering is inconsistent with the real targets.')

fem_by_time = pd.Series(cmp_fem_xel, index=audit_ordered["Timestamp"])
meta_fem_xel = meta["Timestamp"].map(fem_by_time).to_numpy(dtype=float)
raw_fem_corrected_target_rows = []
for split_name, idx in [("val", val_idx), ("test", test_idx)]:
    for convention, fem_pred in [("FEM_as_stored", meta_fem_xel), ("FEM_sign_flipped", -meta_fem_xel)]:
        mask = np.isfinite(fem_pred[idx])
        err = np.abs(y[idx][mask] - fem_pred[idx][mask])
        raw_fem_corrected_target_rows.append({
            "split": split_name, "convention": convention, "count": int(mask.sum()),
            "MAE": float(err.mean()), "within_5_%": float((err <= SPEC_ARCSEC).mean()*100),
            "p90_abs_error": float(np.quantile(err, .90)),
            "pearson": float(np.corrcoef(y[idx][mask], fem_pred[idx][mask])[0, 1]),
        })
raw_fem_corrected_target_df = pd.DataFrame(raw_fem_corrected_target_rows)
display(raw_fem_corrected_target_df)


## Mixture of Experts sul vero XEL


In [ ]:
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import RidgeCV, HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


MOE_MODEL_DIRS = [Path("/content/metrologyml"), Path.cwd()]
MOE_EXPERT_ELEVS = [5, 30, 60, 85]
RUN_DIRECT_MOE_DIAGNOSTIC = True
RUN_UNCONSTRAINED_ROUTER_RETRAIN = True  

def find_first_existing(patterns, dirs=MOE_MODEL_DIRS):
    for d in dirs:
        for p in patterns:
            candidate = d / p
            if candidate.exists():
                return candidate
    return None

def moe_artifact_paths(elev):
    model_path = find_first_existing([
        f"gru_expert_{elev}.h5",
        f"gru_expert_{elev}.keras",
        f"expert_{elev}.h5",
        f"expert_{elev}.keras",
    ])
    scaler_x_path = find_first_existing([
        f"scaler_X_{elev}.pkl",
        f"scaler_x_{elev}.pkl",
        f"scaler_X_{elev}.joblib",
        f"scaler_x_{elev}.joblib",
    ])
    scaler_y_path = find_first_existing([
        f"scaler_y_{elev}.pkl",
        f"scaler_Y_{elev}.pkl",
        f"scaler_y_{elev}.joblib",
        f"scaler_Y_{elev}.joblib",
    ])
    return model_path, scaler_x_path, scaler_y_path

def deterministic_elevation_weights(elev_values, expert_elevs=MOE_EXPERT_ELEVS):
    """Interpolate adjacent elevation experts; use the nearest expert outside their range."""
    elev_values = np.asarray(elev_values, dtype=float).reshape(-1)
    expert_elevs = np.asarray(expert_elevs, dtype=float)
    W = np.zeros((len(elev_values), len(expert_elevs)), dtype=float)
    for i, el in enumerate(elev_values):
        if el <= expert_elevs[0]:
            W[i, 0] = 1.0
        elif el >= expert_elevs[-1]:
            W[i, -1] = 1.0
        else:
            hi = np.searchsorted(expert_elevs, el)
            lo = hi - 1
            t = (el - expert_elevs[lo]) / (expert_elevs[hi] - expert_elevs[lo])
            W[i, lo] = 1.0 - t
            W[i, hi] = t
    return W

if not RUN_DIRECT_MOE_DIAGNOSTIC:
    print('RUN_DIRECT_MOE_DIAGNOSTIC=False: cell skipped.')
else:
    artifact_rows = []
    missing = []
    for elev in MOE_EXPERT_ELEVS:
        model_path, scaler_x_path, scaler_y_path = moe_artifact_paths(elev)
        artifact_rows.append({
            "expert": elev,
            "model": str(model_path) if model_path else None,
            "scaler_X": str(scaler_x_path) if scaler_x_path else None,
            "scaler_y": str(scaler_y_path) if scaler_y_path else None,
        })
        if model_path is None or scaler_x_path is None or scaler_y_path is None:
            missing.append(elev)
    artifacts_df = pd.DataFrame(artifact_rows)
    display(artifacts_df)

    if missing:
        print('Missing expert artifacts:', missing)
        print('Expected files include /content/metrologyml/gru_expert_30.h5, scaler_X_30.pkl and scaler_y_30.pkl.')
    else:
        import joblib
        import tensorflow as tf
        from tensorflow.keras.models import load_model, Model
        from tensorflow.keras.layers import Input, Dense, Multiply, Add, Lambda

        tf.keras.backend.clear_session()
        expert_models = {}
        expert_scalers_X = {}
        expert_scalers_y = {}

        print('Loading FEM experts and scalers...')
        for elev in MOE_EXPERT_ELEVS:
            model_path, scaler_x_path, scaler_y_path = moe_artifact_paths(elev)
            try:
                expert_models[elev] = load_model(
                    model_path,
                    custom_objects={"mae": tf.keras.metrics.MeanAbsoluteError()},
                    compile=False,
                )
            except Exception:
                expert_models[elev] = load_model(
                    model_path,
                    custom_objects={"mae": tf.keras.metrics.MeanAbsoluteError()},
                )
            expert_scalers_X[elev] = joblib.load(scaler_x_path)
            expert_scalers_y[elev] = joblib.load(scaler_y_path)

        print('Predicting expert outputs for clean real samples...')
        X_real = X_seq.astype("float32")
        n_samples, t_steps, n_features = X_real.shape
        expert_pred = {}

        for elev in MOE_EXPERT_ELEVS:
            print(f" -> Expert {elev} deg")
            X_flat = X_real.reshape(-1, n_features)
            X_scaled_flat = expert_scalers_X[elev].transform(X_flat).astype("float32")
            X_scaled = X_scaled_flat.reshape(n_samples, t_steps, n_features)
            pred_scaled = expert_models[elev].predict(X_scaled, verbose=0)
            pred_arcsec = expert_scalers_y[elev].inverse_transform(pred_scaled).reshape(-1)
            expert_pred[elev] = pred_arcsec

        pred_cols = [f"pred_{e}" for e in MOE_EXPERT_ELEVS]
        pred_mat = np.column_stack([expert_pred[e] for e in MOE_EXPERT_ELEVS])
        elev_real = meta["Elevation"].to_numpy(dtype=float)
        y_real = y.astype(float)

        moe_direct_df = meta[["Timestamp", "Elevation", "Target"]].copy()
        for j, col in enumerate(pred_cols):
            moe_direct_df[col] = pred_mat[:, j]
        moe_direct_df["expert_std"] = pred_mat.std(axis=1)
        moe_direct_df["expert_range"] = pred_mat.max(axis=1) - pred_mat.min(axis=1)


        moe_direct_df.rename(columns={"Target": "Target"}).to_csv("moe_real_predictions_from_experts.csv", index=False)
        print('Diagnostic CSV saved: moe_real_predictions_from_experts.csv')

        tr, va, te = np.asarray(train_idx), np.asarray(val_idx), np.asarray(test_idx)
        rows = []

        for j, e in enumerate(MOE_EXPERT_ELEVS):
            rows.append({**eval_predictions(f"expert_{e} / test", y_real[te], pred_mat[te, j]), "kind": "single_expert"})

        pred_mean = pred_mat.mean(axis=1)
        pred_median = np.median(pred_mat, axis=1)
        W_det = deterministic_elevation_weights(elev_real)
        pred_det_interp = np.sum(W_det * pred_mat, axis=1)
        oracle_idx = np.argmin(np.abs(pred_mat - y_real.reshape(-1, 1)), axis=1)
        pred_oracle = pred_mat[np.arange(len(y_real)), oracle_idx]

        rows.extend([
            {**eval_predictions("expert_mean / test", y_real[te], pred_mean[te]), "kind": "ensemble"},
            {**eval_predictions("expert_median / test", y_real[te], pred_median[te]), "kind": "ensemble"},
            {**eval_predictions("deterministic_elevation_interp / test", y_real[te], pred_det_interp[te]), "kind": "physical_router"},
            {**eval_predictions("oracle_best_expert / test", y_real[te], pred_oracle[te]), "kind": "oracle"},
        ])


        X_stack = pd.DataFrame(pred_mat, columns=pred_cols)
        X_stack["elevation"] = elev_real
        X_stack["sin_el"] = np.sin(np.deg2rad(elev_real))
        X_stack["cos_el"] = np.cos(np.deg2rad(elev_real))
        if "Azimuth" in meta.columns:
            az_real = pd.to_numeric(meta["Azimuth"], errors="coerce").to_numpy(dtype=float)
            X_stack["az_sin"] = np.sin(np.deg2rad(az_real))
            X_stack["az_cos"] = np.cos(np.deg2rad(az_real))

        stack_models = {
            "ridge_stack_experts": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", RidgeCV(alphas=np.logspace(-6, 3, 30))),
            ]),
            "huber_stack_experts": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000)),
            ]),
        }
        stack_predictions = {}
        for name, model in stack_models.items():
            m = clone(model)
            m.fit(X_stack.iloc[tr], y_real[tr])
            p = m.predict(X_stack)
            stack_predictions[name] = p
            rows.append({**eval_predictions(f"{name} / val", y_real[va], p[va]), "kind": "calibration"})
            rows.append({**eval_predictions(f"{name} / test", y_real[te], p[te]), "kind": "calibration"})


        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            elev_train_scaled = (elev_real[tr] / 90.0).reshape(-1, 1).astype("float32")
            elev_val_scaled = (elev_real[va] / 90.0).reshape(-1, 1).astype("float32")
            elev_test_scaled = (elev_real[te] / 90.0).reshape(-1, 1).astype("float32")

            elevation_input = Input(shape=(1,), name="elevation_input")
            exp_inputs = [Input(shape=(1,), name=f"pred_expert_{e}") for e in MOE_EXPERT_ELEVS]
            x = Dense(32, activation="relu")(elevation_input)
            x = Dense(16, activation="relu")(x)
            gating_weights = Dense(4, activation="softmax", name="gating_weights")(x)
            weighted = []
            for j, inp in enumerate(exp_inputs):
                wj = Lambda(lambda w, jj=j: w[:, jj:jj+1], name=f"weight_{MOE_EXPERT_ELEVS[j]}")(gating_weights)
                weighted.append(Multiply()([inp, wj]))
            final_prediction = Add(name="final_prediction")(weighted)
            router_model = Model(inputs=[elevation_input] + exp_inputs, outputs=final_prediction)
            router_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss="mae", metrics=["mae"])

            callbacks = [
                tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-5),
            ]
            router_model.fit(
                x=[elev_train_scaled] + [pred_mat[tr, j].reshape(-1, 1).astype("float32") for j in range(4)],
                y=y_real[tr].reshape(-1, 1).astype("float32"),
                validation_data=(
                    [elev_val_scaled] + [pred_mat[va, j].reshape(-1, 1).astype("float32") for j in range(4)],
                    y_real[va].reshape(-1, 1).astype("float32"),
                ),
                batch_size=16,
                epochs=200,
                callbacks=callbacks,
                verbose=0,
            )
            pred_router_test = router_model.predict(
                [elev_test_scaled] + [pred_mat[te, j].reshape(-1, 1).astype("float32") for j in range(4)],
                verbose=0,
            ).reshape(-1)
            pred_router_val = router_model.predict(
                [elev_val_scaled] + [pred_mat[va, j].reshape(-1, 1).astype("float32") for j in range(4)],
                verbose=0,
            ).reshape(-1)
            rows.append({**eval_predictions("unconstrained_router / val", y_real[va], pred_router_val), "kind": "learned_router"})
            rows.append({**eval_predictions("unconstrained_router / test", y_real[te], pred_router_test), "kind": "learned_router"})

            gating_model = Model(inputs=router_model.inputs[0], outputs=router_model.get_layer("gating_weights").output)
            elev_grid = np.linspace(5, 85, 200)
            W_grid = gating_model.predict((elev_grid / 90.0).reshape(-1, 1), verbose=0)
            W_real = gating_model.predict((elev_real / 90.0).reshape(-1, 1), verbose=0)
            for j, e in enumerate(MOE_EXPERT_ELEVS):
                moe_direct_df[f"gate_{e}"] = W_real[:, j]
            moe_direct_df["router_pred_unconstrained"] = np.nan
            moe_direct_df.loc[va, "router_pred_unconstrained"] = pred_router_val
            moe_direct_df.loc[te, "router_pred_unconstrained"] = pred_router_test

            collapse_df = pd.DataFrame({
                "expert": MOE_EXPERT_ELEVS,
                "mean_weight_real": W_real.mean(axis=0),
                "mean_weight_test": W_real[te].mean(axis=0),
                "max_weight_real": W_real.max(axis=0),
                "fraction_weight_gt_0.8_real": (W_real > 0.8).mean(axis=0),
            })
            print('Router mode-collapse diagnostics')
            display(collapse_df)

            plt.figure(figsize=(10, 5))
            colors = ["red", "orange", "green", "blue"]
            for j, e in enumerate(MOE_EXPERT_ELEVS):
                plt.plot(elev_grid, W_grid[:, j], label=f"Expert weight {e} deg", color=colors[j], lw=2.5)
                plt.axvline(e, color="black", linestyle="--", alpha=0.25)
            plt.xlabel('Elevation [deg]')
            plt.ylabel('Router weight')
            plt.title('Elevation-only router: learned weights')
            plt.grid(True, alpha=0.25)
            plt.legend()
            plt.tight_layout()
            plt.show()

        moe_direct_results_df = pd.DataFrame(rows)
        moe_direct_results_df["within_5_%"] = [
            np.nan if not str(row["model"]).endswith("/ test") else
            float((np.abs(y_real[te] - (
                pred_mat[te, int(row["model"].split("_")[1].split(" ")[0] == "30")] if False else y_real[te]
            )) <= SPEC_ARCSEC).mean() * 100)
            for _, row in moe_direct_results_df.iterrows()
        ]

        within_map = {}
        for j, e in enumerate(MOE_EXPERT_ELEVS):
            within_map[f"expert_{e} / test"] = (np.abs(y_real[te] - pred_mat[te, j]) <= SPEC_ARCSEC).mean() * 100
        within_map["expert_mean / test"] = (np.abs(y_real[te] - pred_mean[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["expert_median / test"] = (np.abs(y_real[te] - pred_median[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["deterministic_elevation_interp / test"] = (np.abs(y_real[te] - pred_det_interp[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["oracle_best_expert / test"] = (np.abs(y_real[te] - pred_oracle[te]) <= SPEC_ARCSEC).mean() * 100
        for name, p in stack_predictions.items():
            within_map[f"{name} / test"] = (np.abs(y_real[te] - p[te]) <= SPEC_ARCSEC).mean() * 100
        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            within_map["unconstrained_router / test"] = (np.abs(y_real[te] - pred_router_test) <= SPEC_ARCSEC).mean() * 100
        moe_direct_results_df["within_5_%"] = moe_direct_results_df["model"].map(within_map)

        display(moe_direct_results_df.sort_values("MAE"))

        print('Expert disagreement versus aggregator errors')
        err_corr_rows = []
        aggregators = {
            "mean": pred_mean,
            "deterministic_interp": pred_det_interp,
        }
        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            router_full_pred = np.full_like(y_real, np.nan, dtype=float)
            router_full_pred[va] = pred_router_val
            router_full_pred[te] = pred_router_test
            aggregators["unconstrained_router_val_test"] = router_full_pred
        for name, p in aggregators.items():
            mask = np.isfinite(p)
            err_corr_rows.append({
                "aggregator": name,
                "corr_expert_std_abs_error": float(np.corrcoef(moe_direct_df.loc[mask, "expert_std"], np.abs(y_real[mask] - p[mask]))[0, 1]),
                "corr_expert_range_abs_error": float(np.corrcoef(moe_direct_df.loc[mask, "expert_range"], np.abs(y_real[mask] - p[mask]))[0, 1]),
            })
        display(pd.DataFrame(err_corr_rows))

        print('Worst test cases for the deterministic aggregator')
        worst_moe = meta.iloc[te][["Timestamp", "Elevation", "Target"]].copy()
        worst_moe["pred_det_interp"] = pred_det_interp[te]
        worst_moe["abs_error_det_interp"] = np.abs(y_real[te] - pred_det_interp[te])
        worst_moe["expert_std"] = moe_direct_df.iloc[te]["expert_std"].to_numpy()
        display(worst_moe.sort_values("abs_error_det_interp", ascending=False).head(15))


In [ ]:
if "pred_mat" not in globals():
    print('Expert predictions are unavailable; run the direct MoE cell.')
else:
    X_moe_base = pd.DataFrame(pred_mat, columns=pred_cols)
    X_moe_base["elevation"] = meta["Elevation"].to_numpy(dtype=float)
    X_moe_base["sin_el"] = np.sin(np.deg2rad(X_moe_base["elevation"]))
    X_moe_base["cos_el"] = np.cos(np.deg2rad(X_moe_base["elevation"]))
    X_moe_rx = pd.concat([X_moe_base.reset_index(drop=True), receiver_one_hot(meta)], axis=1)
    moe_rx_rows = []
    moe_rx_predictions = {}
    for feature_set, Xuse in [("without_receiver", X_moe_base), ("with_receiver", X_moe_rx)]:
        for model_name, estimator in stack_models.items():
            fitted = clone(estimator).fit(Xuse.iloc[train_idx], y[train_idx])
            for split_name, idx in [("val", val_idx), ("test", test_idx)]:
                pred = fitted.predict(Xuse.iloc[idx])
                moe_rx_predictions[(feature_set, model_name, split_name)] = pred
                row = eval_predictions(f"{feature_set}:{model_name} / {split_name}", y[idx], pred)
                row.update({
                    "feature_set": feature_set, "base_model": model_name, "split": split_name,
                    "within_5_%": float((np.abs(y[idx]-pred) <= SPEC_ARCSEC).mean()*100),
                })
                moe_rx_rows.append(row)
    moe_receiver_ablation_df = pd.DataFrame(moe_rx_rows).sort_values(["split", "MAE"])
    display(moe_receiver_ablation_df)


## Rivalutazione del domain shift dopo la correzione XEL


In [ ]:
EXTREME_TARGET_ARCSEC = 15.0
SHIFT_NN_NEIGHBORS = 25

def aggregate_sensor_state(values):
    values = np.asarray(values, dtype=float)
    q10, q25, q50, q75, q90 = np.nanpercentile(values, [10, 25, 50, 75, 90], axis=1)
    return pd.DataFrame({
        "temp_mean": np.nanmean(values, axis=1),
        "temp_std_spatial": np.nanstd(values, axis=1),
        "temp_min": np.nanmin(values, axis=1),
        "temp_max": np.nanmax(values, axis=1),
        "temp_range": np.nanmax(values, axis=1)-np.nanmin(values, axis=1),
        "temp_p10": q10, "temp_p50": q50, "temp_p90": q90, "temp_iqr": q75-q25,
    })

def infer_fem_columns(df):
    out = normalize_sensor_columns(df.copy())
    sensors = get_sensor_cols(out)
    elev_col = next((c for c in ["Elevation", "elevation", "ElevationAngle"] if c in out.columns), None)
    target_col = next((c for c in ["XEL", "XEl Offset", "XEL_offset"] if c in out.columns), None)
    if target_col is None and "Accuracy_1" in out.columns and elev_col is not None:
        out["XEL"] = pd.to_numeric(out["Accuracy_1"], errors="coerce") * np.cos(
            np.deg2rad(pd.to_numeric(out[elev_col], errors="coerce"))
        )
        target_col = "XEL"
    return out, sensors, elev_col, target_col


In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors

SHIFT_V2_MAX_FEM_ROWS = 160_000
SHIFT_V2_TARGET_PER_ELEV = 40_000
SHIFT_V2_EXPERT_ELEVS = [5, 30, 60, 85]
SHIFT_V2_RANDOM_SEED = RANDOM_SEED if "RANDOM_SEED" in globals() else 42

def load_fem_stratified_for_shift_v2():
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        print('FEM file not found.')
        return None, None, None, None, None

    print('Loading FEM data stratified by elevation:', fem_path)
    suffix = Path(fem_path).suffix.lower()

    if suffix == ".csv":


        fem_all = pd.read_csv(fem_path)
    else:
        fem_all = pd.read_excel(fem_path)

    fem_all, fem_sensor_cols, fem_elev_col, fem_target_col = infer_fem_columns(fem_all)
    if fem_elev_col is None:
        print('FEM elevation column not found; using a global random sample.')
        fem_sample = fem_all.sample(n=min(SHIFT_V2_MAX_FEM_ROWS, len(fem_all)), random_state=SHIFT_V2_RANDOM_SEED)
    else:
        parts = []
        for elev, g in fem_all.groupby(fem_elev_col):
            n_take = min(SHIFT_V2_TARGET_PER_ELEV, len(g))
            parts.append(g.sample(n=n_take, random_state=SHIFT_V2_RANDOM_SEED))
        fem_sample = pd.concat(parts, ignore_index=True)
        if len(fem_sample) > SHIFT_V2_MAX_FEM_ROWS:
            fem_sample = fem_sample.sample(n=SHIFT_V2_MAX_FEM_ROWS, random_state=SHIFT_V2_RANDOM_SEED)

    print("FEM sample v2:", len(fem_sample))
    if fem_elev_col is not None:
        display(fem_sample[fem_elev_col].value_counts().sort_index().rename_axis("Elevation").reset_index(name="count"))
    print("FEM sensor cols:", len(fem_sensor_cols), "| elev:", fem_elev_col, "| target:", fem_target_col)
    return fem_sample, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col

def build_shift_v2_tables():
    fem_sample, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col = load_fem_stratified_for_shift_v2()
    if fem_sample is None or len(fem_sensor_cols) < 20:
        return None


    comparable_cols = [
        "temp_mean", "temp_std_spatial", "temp_min", "temp_max", "temp_range",
        "temp_p10", "temp_p50", "temp_p90", "temp_iqr", "elevation"
    ]

    real_current = X_seq[:, -1, :]
    real_v2 = aggregate_sensor_state(real_current)
    real_v2["elevation"] = meta["Elevation"].to_numpy(dtype=float)
    real_v2["target"] = y.astype(float)
    real_v2["abs_target"] = np.abs(y.astype(float))
    real_v2["domain"] = "real"
    real_v2["Timestamp"] = meta["Timestamp"].to_numpy()

    fem_arr = fem_sample[fem_sensor_cols].to_numpy(dtype="float32")
    fem_v2 = aggregate_sensor_state(fem_arr)
    fem_v2["elevation"] = pd.to_numeric(fem_sample[fem_elev_col], errors="coerce").to_numpy(dtype=float) if fem_elev_col else np.nan
    fem_v2["target"] = pd.to_numeric(fem_sample[fem_target_col], errors="coerce").to_numpy(dtype=float) if fem_target_col else np.nan
    fem_v2["abs_target"] = np.abs(fem_v2["target"].to_numpy(dtype=float))
    fem_v2["domain"] = "fem"
    fem_v2["Timestamp"] = pd.NaT

    return {
        "real_v2": real_v2,
        "fem_v2": fem_v2,
        "feature_cols_v2": comparable_cols,
        "fem_target_col": fem_target_col,
        "fem_path": fem_path,
    }

shift_v2 = build_shift_v2_tables()
if shift_v2 is not None:
    real_v2 = shift_v2["real_v2"]
    fem_v2 = shift_v2["fem_v2"]
    feature_cols_v2 = shift_v2["feature_cols_v2"]
    print('Comparable v2 features:', feature_cols_v2)
    display(pd.concat([
        real_v2[feature_cols_v2 + ["target"]].describe(percentiles=[.1, .5, .9]).assign(domain_summary="real"),
        fem_v2[feature_cols_v2 + ["target"]].describe(percentiles=[.1, .5, .9]).assign(domain_summary="fem"),
    ]))


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Shift v2 is unavailable.')
else:
    rng = np.random.default_rng(SHIFT_V2_RANDOM_SEED)
    n_real = len(real_v2)
    fem_idx = rng.choice(len(fem_v2), size=min(len(fem_v2), n_real * 3), replace=False)
    domain_v2_df = pd.concat([real_v2, fem_v2.iloc[fem_idx]], ignore_index=True)
    y_domain_v2 = (domain_v2_df["domain"] == "real").astype(int).to_numpy()
    X_domain_v2 = domain_v2_df[feature_cols_v2]

    Xtr, Xte, ytr, yte = train_test_split(
        X_domain_v2, y_domain_v2, test_size=0.30, stratify=y_domain_v2, random_state=SHIFT_V2_RANDOM_SEED
    )

    domain_clf_v2 = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=600,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SHIFT_V2_RANDOM_SEED,
            n_jobs=-1,
        )),
    ])
    domain_clf_v2.fit(Xtr, ytr)
    proba = domain_clf_v2.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"Domain classifier v2 FEM-vs-Real: AUC={roc_auc_score(yte, proba):.3f} | accuracy={accuracy_score(yte, pred):.3f}")
    print(classification_report(yte, pred, target_names=["FEM", "Real"]))

    perm_v2 = permutation_importance(
        domain_clf_v2, Xte, yte,
        n_repeats=20,
        random_state=SHIFT_V2_RANDOM_SEED,
        scoring="roc_auc",
        n_jobs=-1,
    )
    domain_importance_v2_df = pd.DataFrame({
        "feature": feature_cols_v2,
        "importance_mean_auc_drop": perm_v2.importances_mean,
        "importance_std": perm_v2.importances_std,
    }).sort_values("importance_mean_auc_drop", ascending=False)
    display(domain_importance_v2_df)

    real_v2["domain_score_real_proba_v2"] = domain_clf_v2.predict_proba(real_v2[feature_cols_v2])[:, 1]
    fem_v2["domain_score_real_proba_v2"] = domain_clf_v2.predict_proba(fem_v2[feature_cols_v2])[:, 1]
    display(pd.DataFrame({
        "real": real_v2["domain_score_real_proba_v2"].describe(percentiles=[.1, .25, .5, .75, .9]),
        "fem": fem_v2["domain_score_real_proba_v2"].describe(percentiles=[.1, .25, .5, .75, .9]),
    }))

    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    axs[0].hist(fem_v2["domain_score_real_proba_v2"], bins=50, alpha=0.55, density=True, label="FEM")
    axs[0].hist(real_v2["domain_score_real_proba_v2"], bins=50, alpha=0.55, density=True, label="Real")
    axs[0].set_xlabel('P(real), comparable features')
    axs[0].set_title("Domain classifier v2")
    axs[0].grid(True, alpha=0.25)
    axs[0].legend()

    top = domain_importance_v2_df.head(10).iloc[::-1]
    axs[1].barh(top["feature"], top["importance_mean_auc_drop"])
    axs[1].set_xlabel('Permutation AUC drop')
    axs[1].set_title('Input-shift features v2')
    axs[1].grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Shift v2 is unavailable.')
else:
    support_pipe_v2 = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    fem_scaled_v2 = support_pipe_v2.fit_transform(fem_v2[feature_cols_v2])
    real_scaled_v2 = support_pipe_v2.transform(real_v2[feature_cols_v2])

    nn_v2 = NearestNeighbors(n_neighbors=min(SHIFT_NN_NEIGHBORS, len(fem_scaled_v2))).fit(fem_scaled_v2)
    dist_v2, idx_v2 = nn_v2.kneighbors(real_scaled_v2)
    real_v2["fem_nn_dist_1_v2"] = dist_v2[:, 0]
    real_v2["fem_nn_dist_med_v2"] = np.median(dist_v2, axis=1)


    rng = np.random.default_rng(SHIFT_V2_RANDOM_SEED)
    ref_idx = rng.choice(len(fem_scaled_v2), size=min(20_000, len(fem_scaled_v2)), replace=False)
    nn_self_v2 = NearestNeighbors(n_neighbors=2).fit(fem_scaled_v2)
    self_dist_v2, _ = nn_self_v2.kneighbors(fem_scaled_v2[ref_idx])
    fem_neighbor_dist_v2 = self_dist_v2[:, 1]
    thr95_v2 = np.quantile(fem_neighbor_dist_v2, 0.95)
    thr99_v2 = np.quantile(fem_neighbor_dist_v2, 0.99)
    real_v2["support_region_v2"] = np.select(
        [
            real_v2["fem_nn_dist_1_v2"] <= thr95_v2,
            real_v2["fem_nn_dist_1_v2"] <= thr99_v2,
        ],
        ["inside_fem_p95", "border_fem_p99"],
        default="outside_fem_p99",
    )
    print(f"Support thresholds v2: p95={thr95_v2:.3f}, p99={thr99_v2:.3f}")
    support_v2_summary = real_v2.groupby("support_region_v2").agg(
        count=("target", "size"),
        coverage_pct=("target", lambda s: len(s) / len(real_v2) * 100),
        median_abs_target=("abs_target", "median"),
        p90_abs_target=("abs_target", lambda s: s.quantile(.90)),
        extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
        median_domain_score=("domain_score_real_proba_v2", "median") if "domain_score_real_proba_v2" in real_v2.columns else ("abs_target", "median"),
    ).reset_index()
    display(support_v2_summary)


    if fem_v2["target"].notna().sum() >= 100:
        fem_targets_nn_v2 = fem_v2["target"].to_numpy(dtype=float)[idx_v2]
        real_v2["fem_nn_target_median_v2"] = np.nanmedian(fem_targets_nn_v2, axis=1)
        real_v2["conditional_residual_median_v2"] = real_v2["target"] - real_v2["fem_nn_target_median_v2"]
        real_v2["conditional_abs_residual_median_v2"] = real_v2["conditional_residual_median_v2"].abs()

        conditional_v2_summary = real_v2.groupby("support_region_v2").agg(
            count=("target", "size"),
            mae_fem_neighbor_median=("conditional_abs_residual_median_v2", "mean"),
            median_abs_residual=("conditional_abs_residual_median_v2", "median"),
            p90_abs_residual=("conditional_abs_residual_median_v2", lambda s: s.quantile(.90)),
            extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
        ).reset_index()
        print("Conditional shift v2")
        display(conditional_v2_summary)

        print('Largest conditional residuals v2')
        show_cols = ["Timestamp", "elevation", "target", "abs_target", "fem_nn_target_median_v2", "conditional_residual_median_v2", "fem_nn_dist_1_v2", "support_region_v2"]
        display(real_v2.sort_values("conditional_abs_residual_median_v2", ascending=False).head(20)[show_cols])

        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        axs[0].scatter(real_v2["fem_nn_target_median_v2"], real_v2["target"], c=real_v2["abs_target"], cmap="magma", s=28, alpha=0.8)
        lim = np.nanmax(np.abs(real_v2[["fem_nn_target_median_v2", "target"]].to_numpy()))
        axs[0].plot([-lim, lim], [-lim, lim], "k--", lw=1)
        axs[0].set_xlabel('Local FEM target v2 [arcsec]')
        axs[0].set_ylabel('Observed target [arcsec]')
        axs[0].set_title('P(y|X): local FEM versus real')
        axs[0].grid(True, alpha=0.25)

        axs[1].scatter(real_v2["fem_nn_dist_1_v2"], real_v2["conditional_abs_residual_median_v2"], c=real_v2["abs_target"], cmap="magma", s=28, alpha=0.8)
        axs[1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
        axs[1].set_xlabel('FEM distance v2')
        axs[1].set_ylabel('|Conditional residual| [arcsec]')
        axs[1].set_title("Support distance vs conditional residual")
        axs[1].grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
    else:
        print('FEM targets are unavailable for conditional shift v2.')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

if "shift_v2" not in globals() or shift_v2 is None:
    print('Shift v2 is unavailable.')
else:
    for frame in (real_v2, fem_v2):
        frame["temp_p10_centered"] = frame["temp_p10"] - frame["temp_mean"]
        frame["temp_p50_centered"] = frame["temp_p50"] - frame["temp_mean"]
        frame["temp_p90_centered"] = frame["temp_p90"] - frame["temp_mean"]

    shift_feature_sets = {
        "full_reference": feature_cols_v2,
        "without_iqr": [c for c in feature_cols_v2 if c != "temp_iqr"],
        "thermal_without_iqr_elevation": [c for c in feature_cols_v2 if c not in ["temp_iqr", "elevation"]],
        "thermal_level_only": ["temp_mean", "temp_min", "temp_max", "temp_p10", "temp_p50", "temp_p90"],
        "thermal_shape_centered": [
            "temp_std_spatial", "temp_range", "temp_p10_centered", "temp_p50_centered", "temp_p90_centered"
        ],
    }

    rng = np.random.default_rng(SHIFT_V2_RANDOM_SEED)
    fem_domain_idx = rng.choice(len(fem_v2), size=min(len(fem_v2), len(real_v2) * 3), replace=False)
    domain_ablation_rows = []
    for set_name, cols in shift_feature_sets.items():
        combined = pd.concat([real_v2[cols], fem_v2.iloc[fem_domain_idx][cols]], ignore_index=True)
        labels = np.r_[np.ones(len(real_v2), dtype=int), np.zeros(len(fem_domain_idx), dtype=int)]
        Xtr, Xte, ytr, yte = train_test_split(
            combined, labels, test_size=.30, stratify=labels, random_state=SHIFT_V2_RANDOM_SEED
        )
        for classifier_name, classifier in [
            ("logistic", LogisticRegression(max_iter=3000, class_weight="balanced")),
            ("random_forest", RandomForestClassifier(
                n_estimators=400, min_samples_leaf=5, max_features="sqrt",
                class_weight="balanced", random_state=SHIFT_V2_RANDOM_SEED, n_jobs=-1,
            )),
        ]:
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", classifier),
            ])
            pipe.fit(Xtr, ytr)
            score = pipe.predict_proba(Xte)[:, 1]
            domain_ablation_rows.append({
                "feature_set": set_name, "classifier": classifier_name, "n_features": len(cols),
                "AUC": float(roc_auc_score(yte, score)), "drift_strength": float(2 * abs(roc_auc_score(yte, score) - .5)),
            })
    domain_ablation_df = pd.DataFrame(domain_ablation_rows).sort_values(["feature_set", "classifier"])
    display(domain_ablation_df)


    matched_feature_sets = {
        "thermal_without_iqr": shift_feature_sets["thermal_without_iqr_elevation"],
        "thermal_level_only": shift_feature_sets["thermal_level_only"],
        "thermal_shape_centered": shift_feature_sets["thermal_shape_centered"],
    }
    fem_elev_levels = np.sort(fem_v2["elevation"].dropna().unique())
    real_nearest_elev = fem_elev_levels[np.abs(real_v2["elevation"].to_numpy()[:, None] - fem_elev_levels[None, :]).argmin(axis=1)]
    matched_support_rows = []
    matched_conditional_rows = []
    conditional_records = []

    for set_name, cols in matched_feature_sets.items():
        all_real_dist = np.full(len(real_v2), np.nan)
        all_real_inside95 = np.zeros(len(real_v2), dtype=bool)
        all_real_inside99 = np.zeros(len(real_v2), dtype=bool)
        all_local_target = np.full(len(real_v2), np.nan)
        threshold_rows = []
        for elev in fem_elev_levels:
            fi = np.flatnonzero(np.isclose(fem_v2["elevation"].to_numpy(dtype=float), elev))
            ri = np.flatnonzero(np.isclose(real_nearest_elev, elev))
            if len(fi) < 200 or len(ri) == 0:
                continue
            shuffled = rng.permutation(fi)
            n_cal = min(5000, max(500, len(shuffled) // 5))
            cal_idx = shuffled[:n_cal]
            ref_idx = shuffled[n_cal:]
            if len(ref_idx) > 25000:
                ref_idx = rng.choice(ref_idx, size=25000, replace=False)

            preprocess = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
            ref_scaled = preprocess.fit_transform(fem_v2.iloc[ref_idx][cols])
            cal_scaled = preprocess.transform(fem_v2.iloc[cal_idx][cols])
            real_scaled = preprocess.transform(real_v2.iloc[ri][cols])
            k = min(25, len(ref_idx))
            nn = NearestNeighbors(n_neighbors=k).fit(ref_scaled)
            cal_dist, _ = nn.kneighbors(cal_scaled)
            real_dist, real_nn_idx = nn.kneighbors(real_scaled)
            cal_kdist = cal_dist[:, min(4, k - 1)]
            real_kdist = real_dist[:, min(4, k - 1)]
            p95, p99 = np.quantile(cal_kdist, [.95, .99])
            all_real_dist[ri] = real_kdist
            all_real_inside95[ri] = real_kdist <= p95
            all_real_inside99[ri] = real_kdist <= p99
            neighbor_targets = fem_v2.iloc[ref_idx]["target"].to_numpy(dtype=float)[real_nn_idx]
            all_local_target[ri] = np.nanmedian(neighbor_targets, axis=1)
            threshold_rows.append((elev, len(ri), p95, p99))

        valid = np.isfinite(all_real_dist) & np.isfinite(all_local_target)
        residual = real_v2["target"].to_numpy(dtype=float) - all_local_target
        corr = np.corrcoef(real_v2.loc[valid, "target"], all_local_target[valid])[0, 1] if valid.sum() > 2 else np.nan
        matched_support_rows.append({
            "feature_set": set_name, "count": int(valid.sum()),
            "inside_matched_fem_p95_percent": float(all_real_inside95[valid].mean() * 100),
            "inside_matched_fem_p99_percent": float(all_real_inside99[valid].mean() * 100),
            "median_matched_k5_distance": float(np.nanmedian(all_real_dist[valid])),
            "p90_matched_k5_distance": float(np.nanquantile(all_real_dist[valid], .90)),
        })
        matched_conditional_rows.append({
            "feature_set": set_name, "count": int(valid.sum()),
            "pearson_real_vs_local_FEM": float(corr),
            "MAE_local_FEM": float(np.nanmean(np.abs(residual[valid]))),
            "median_abs_residual": float(np.nanmedian(np.abs(residual[valid]))),
            "p90_abs_residual": float(np.nanquantile(np.abs(residual[valid]), .90)),
        })
        for idx in np.flatnonzero(valid):
            conditional_records.append({
                "feature_set": set_name, "Timestamp": real_v2.iloc[idx]["Timestamp"],
                "real_elevation": real_v2.iloc[idx]["elevation"], "matched_FEM_elevation": real_nearest_elev[idx],
                "target_real": real_v2.iloc[idx]["target"], "target_local_FEM": all_local_target[idx],
                "conditional_residual": residual[idx], "abs_conditional_residual": abs(residual[idx]),
                "matched_k5_distance": all_real_dist[idx],
                "inside_p95": all_real_inside95[idx], "inside_p99": all_real_inside99[idx],
            })

    matched_support_ablation_df = pd.DataFrame(matched_support_rows)
    matched_conditional_shift_df = pd.DataFrame(matched_conditional_rows)
    matched_conditional_records_df = pd.DataFrame(conditional_records)
    display(matched_support_ablation_df)
    display(matched_conditional_shift_df)
    print('Largest residuals after elevation matching without temp_iqr')
    display(matched_conditional_records_df[
        matched_conditional_records_df["feature_set"] == "thermal_without_iqr"
    ].sort_values("abs_conditional_residual", ascending=False).head(20))

    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    auc_plot = domain_ablation_df[domain_ablation_df["classifier"] == "logistic"]
    axs[0].barh(auc_plot["feature_set"], auc_plot["AUC"], color="tab:orange", alpha=.8)
    axs[0].axvline(.5, color="black", linestyle="--", linewidth=1)
    axs[0].set_xlim(.45, 1.01)
    axs[0].set_xlabel("AUC FEM vs Real")
    axs[0].set_title('Separability after ablations')
    axs[0].grid(True, axis="x", alpha=.25)
    axs[1].barh(matched_support_ablation_df["feature_set"], matched_support_ablation_df["inside_matched_fem_p99_percent"], color="tab:blue", alpha=.8)
    axs[1].set_xlabel('Real samples inside FEM p99 support [%]')
    axs[1].set_title('Thermal support at matched elevation')
    axs[1].grid(True, axis="x", alpha=.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "matched_conditional_records_df" not in globals() or matched_conditional_records_df.empty:
    print('Conditional-shift records are unavailable; run the preceding cell.')
else:
    support_conditional_records = matched_conditional_records_df.copy()
    support_conditional_records["support_region"] = np.select(
        [support_conditional_records["inside_p95"], support_conditional_records["inside_p99"]],
        ["inside_p95", "border_p95_p99"], default="outside_p99"
    )
    split_by_time = pd.Series("train", index=meta["Timestamp"])
    split_by_time.loc[meta.iloc[val_idx]["Timestamp"]] = "val"
    split_by_time.loc[meta.iloc[test_idx]["Timestamp"]] = "test"
    support_conditional_records["split"] = support_conditional_records["Timestamp"].map(split_by_time)
    train_median_target = float(np.median(y_train))

    def summarize_conditional_group(g, region_label):
        target = g["target_real"].to_numpy(dtype=float)
        local = g["target_local_FEM"].to_numpy(dtype=float)
        corr = np.corrcoef(target, local)[0, 1] if len(g) >= 3 and np.std(target) > 0 and np.std(local) > 0 else np.nan
        mae_fem = np.mean(np.abs(target - local)) if len(g) else np.nan
        mae_zero = np.mean(np.abs(target)) if len(g) else np.nan
        mae_median = np.mean(np.abs(target - train_median_target)) if len(g) else np.nan
        return {
            "feature_set": g["feature_set"].iloc[0], "support_region": region_label,
            "count": len(g), "pearson_real_vs_local_FEM": corr,
            "MAE_local_FEM": mae_fem, "MAE_zero": mae_zero, "MAE_train_median": mae_median,
            "delta_MAE_FEM_minus_train_median": mae_fem - mae_median,
            "within_5_local_FEM_percent": float((np.abs(target-local) <= SPEC_ARCSEC).mean()*100) if len(g) else np.nan,
            "p90_abs_residual": float(np.quantile(np.abs(target-local), .90)) if len(g) else np.nan,
        }

    rows = []
    for feature_set, fs in support_conditional_records.groupby("feature_set", observed=True):
        for region, group in fs.groupby("support_region", observed=True):
            rows.append(summarize_conditional_group(group, region))
        inside_p99_group = fs[fs["inside_p99"]]
        if len(inside_p99_group):
            rows.append(summarize_conditional_group(inside_p99_group, "inside_p99_combined"))
        rows.append(summarize_conditional_group(fs, "all"))
    conditional_by_support_df = pd.DataFrame(rows).sort_values(["feature_set", "support_region"])
    display(conditional_by_support_df)


    eval_rows = []
    eval_records = support_conditional_records[support_conditional_records["split"].isin(["val", "test"])]
    for (feature_set, split), fs in eval_records.groupby(["feature_set", "split"], observed=True):
        for region, group in [("inside_p99_combined", fs[fs["inside_p99"]]), ("outside_p99", fs[~fs["inside_p99"]])]:
            if len(group):
                row = summarize_conditional_group(group, region)
                row["split"] = split
                eval_rows.append(row)
    conditional_by_support_eval_df = pd.DataFrame(eval_rows)
    display(conditional_by_support_eval_df)


In [ ]:
SRT_LATITUDE_DEG = 39 + 29/60 + 32.2/3600
SRT_LONGITUDE_DEG = 9 + 14/60 + 46.8/3600

def solar_position_utc(timestamps, latitude_deg=SRT_LATITUDE_DEG, longitude_deg=SRT_LONGITUDE_DEG):
    ts = pd.to_datetime(pd.Series(timestamps), utc=True)
    unix_seconds = ts.astype("int64").to_numpy(dtype=float) / 1e9
    julian_day = unix_seconds / 86400.0 + 2440587.5
    century = (julian_day - 2451545.0) / 36525.0
    mean_long = np.mod(280.46646 + century*(36000.76983 + century*0.0003032), 360.0)
    mean_anomaly = 357.52911 + century*(35999.05029 - 0.0001537*century)
    eccentricity = 0.016708634 - century*(0.000042037 + 0.0000001267*century)
    anomaly_rad = np.deg2rad(mean_anomaly)
    center = (
        np.sin(anomaly_rad)*(1.914602-century*(0.004817+0.000014*century))
        + np.sin(2*anomaly_rad)*(0.019993-0.000101*century) + np.sin(3*anomaly_rad)*0.000289
    )
    true_long = mean_long + center
    omega = 125.04 - 1934.136*century
    apparent_long = true_long - 0.00569 - 0.00478*np.sin(np.deg2rad(omega))
    mean_obliquity = 23 + (26 + (21.448-century*(46.815+century*(0.00059-century*0.001813)))/60)/60
    obliquity = mean_obliquity + 0.00256*np.cos(np.deg2rad(omega))
    obliquity_rad = np.deg2rad(obliquity)
    declination = np.arcsin(np.sin(obliquity_rad)*np.sin(np.deg2rad(apparent_long)))
    y_term = np.tan(obliquity_rad/2)**2
    mlr = np.deg2rad(mean_long)
    equation_time = 4*np.rad2deg(
        y_term*np.sin(2*mlr) - 2*eccentricity*np.sin(anomaly_rad)
        + 4*eccentricity*y_term*np.sin(anomaly_rad)*np.cos(2*mlr)
        - .5*y_term**2*np.sin(4*mlr) - 1.25*eccentricity**2*np.sin(2*anomaly_rad)
    )
    utc_minutes = ts.dt.hour.to_numpy()*60 + ts.dt.minute.to_numpy() + ts.dt.second.to_numpy()/60
    solar_minutes = np.mod(utc_minutes + equation_time + 4*longitude_deg, 1440.0)
    hour_angle = np.deg2rad(solar_minutes/4 - 180)
    latitude = np.deg2rad(latitude_deg)
    cos_zenith = np.clip(
        np.sin(latitude)*np.sin(declination) + np.cos(latitude)*np.cos(declination)*np.cos(hour_angle), -1, 1
    )
    elevation = 90 - np.rad2deg(np.arccos(cos_zenith))
    azimuth = np.mod(np.rad2deg(np.arctan2(
        np.sin(hour_angle), np.cos(hour_angle)*np.sin(latitude)-np.tan(declination)*np.cos(latitude)
    )) + 180, 360)
    return pd.DataFrame({"solar_elevation": elevation, "solar_azimuth": azimuth})

solar_all = solar_position_utc(meta["Timestamp"])
cluster_audit_df = meta.copy().reset_index(drop=True)
cluster_audit_df["split"] = "train"
cluster_audit_df.loc[val_idx, "split"] = "val"
cluster_audit_df.loc[test_idx, "split"] = "test"
cluster_audit_df["solar_elevation"] = solar_all["solar_elevation"].to_numpy()
cluster_audit_df["solar_azimuth"] = solar_all["solar_azimuth"].to_numpy()
cluster_audit_df["sun_up"] = cluster_audit_df["solar_elevation"] > 0
cluster_audit_df["relative_sun_azimuth"] = (
    cluster_audit_df["Azimuth"] - cluster_audit_df["solar_azimuth"] + 180
) % 360 - 180
cluster_audit_df["current_temp_mean"] = np.mean(X_seq[:, -1, :], axis=1)
cluster_audit_df["current_temp_std"] = np.std(X_seq[:, -1, :], axis=1)
cluster_audit_df["FEM_XEL"] = meta_fem_xel if "meta_fem_xel" in globals() else np.nan
if "pred_mat" in globals():
    cluster_audit_df["expert_std"] = np.std(pred_mat, axis=1)
cluster_audit_df["cluster_25_27_june"] = (
    (cluster_audit_df["Timestamp"] >= pd.Timestamp("2025-06-25", tz=None)) &
    (cluster_audit_df["Timestamp"] < pd.Timestamp("2025-06-28", tz=None))
)
cluster_detail_df = cluster_audit_df[cluster_audit_df["cluster_25_27_june"]].copy()
cluster_detail_df["abs_target"] = cluster_detail_df["Target"].abs()
cluster_detail_df["FEM_residual"] = cluster_detail_df["Target"] - cluster_detail_df["FEM_XEL"]
cluster_detail_df["dt_previous_min"] = cluster_detail_df["Timestamp"].diff().dt.total_seconds()/60
cluster_cols = [
    "Timestamp", "split", "ReceiverCode", "ReceiverBand", "Elevation", "Azimuth", "Target", "FEM_XEL",
    "FEM_residual", "FitXelSigmaArcsec", "LonFitHPBWArcsec", "solar_elevation", "solar_azimuth",
    "relative_sun_azimuth", "sun_up", "current_temp_mean", "current_temp_std", "expert_std", "dt_previous_min",
]
cluster_cols = [c for c in cluster_cols if c in cluster_detail_df.columns]
display(cluster_detail_df.sort_values("Timestamp")[cluster_cols])

cluster_context_df = cluster_audit_df.groupby("cluster_25_27_june").agg(
    count=("Target", "size"), median_target=("Target", "median"), median_abs_target=("Target", lambda s: s.abs().median()),
    target_gt10_percent=("Target", lambda s: (s.abs()>10).mean()*100),
    median_elevation=("Elevation", "median"), median_fit_sigma=("FitXelSigmaArcsec", "median"),
    median_HPBW=("LonFitHPBWArcsec", "median"), sun_up_percent=("sun_up", lambda s: s.mean()*100),
    median_solar_elevation=("solar_elevation", "median"), median_temp_mean=("current_temp_mean", "median"),
    median_temp_std=("current_temp_std", "median"),
).reset_index()
display(cluster_context_df)
display(cluster_detail_df.groupby(["ReceiverBand", "sun_up"], observed=True).agg(
    count=("Target", "size"), median_target=("Target", "median"), median_abs_target=("Target", lambda s: s.abs().median()),
    median_FEM=("FEM_XEL", "median"), median_FEM_residual=("FEM_residual", "median"),
).reset_index())


In [ ]:
if "pred_mat" not in globals():
    print('Expert predictions are unavailable.')
else:
    oracle_idx_all = np.argmin(np.abs(pred_mat - y.reshape(-1, 1)), axis=1)
    oracle_expert_all = np.asarray(MOE_EXPERT_ELEVS, dtype=int)[oracle_idx_all]
    oracle_audit_df = meta.copy().reset_index(drop=True)
    oracle_audit_df["oracle_expert"] = oracle_expert_all
    oracle_audit_df["oracle_abs_error"] = np.min(np.abs(pred_mat-y.reshape(-1, 1)), axis=1)
    oracle_audit_df["expert_std"] = np.std(pred_mat, axis=1)
    oracle_audit_df["elev_band"] = pd.cut(oracle_audit_df["Elevation"], [0,20,30,45,60,75,90], include_lowest=True)
    train_temp_q = np.quantile(np.mean(X_seq[train_idx, -1, :], axis=1), [.25,.5,.75])
    oracle_audit_df["thermal_level"] = pd.cut(
        np.mean(X_seq[:, -1, :], axis=1), [-np.inf, *np.unique(train_temp_q), np.inf], include_lowest=True
    )

    def oracle_distribution(group_col):
        counts = oracle_audit_df.groupby([group_col, "oracle_expert"], observed=True).size().rename("count").reset_index()
        counts["group_total"] = counts.groupby(group_col, observed=True)["count"].transform("sum")
        counts["percent"] = counts["count"] / counts["group_total"] * 100
        return counts

    oracle_by_elevation_df = oracle_distribution("elev_band")
    oracle_by_receiver_df = oracle_distribution("ReceiverBand")
    oracle_by_thermal_regime_df = oracle_distribution("thermal_level")
    display(oracle_by_elevation_df)
    display(oracle_by_receiver_df)
    display(oracle_by_thermal_regime_df)


    F_oracle = fb.transform(X_seq, X_elev, meta).reset_index(drop=True)
    F_oracle_rx = pd.concat([F_oracle, receiver_one_hot(meta)], axis=1)
    oracle_router_rows = []
    oracle_router_predictions = {}
    for feature_set, Xuse in [("physical", F_oracle), ("physical_plus_receiver", F_oracle_rx)]:
        selector = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=600, min_samples_leaf=8, max_features="sqrt",
                class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
            )),
        ])
        selector.fit(Xuse.iloc[train_idx], oracle_idx_all[train_idx])
        for split_name, idx in [("val", val_idx), ("test", test_idx)]:
            selected_idx = selector.predict(Xuse.iloc[idx]).astype(int)
            selected_pred = pred_mat[idx, selected_idx]
            oracle_router_predictions[(feature_set, split_name)] = selected_pred
            err = np.abs(y[idx]-selected_pred)
            oracle_router_rows.append({
                "feature_set": feature_set, "split": split_name, "count": len(idx),
                "oracle_class_accuracy_percent": float((selected_idx==oracle_idx_all[idx]).mean()*100),
                "selected_expert_MAE": float(err.mean()), "within_5_percent": float((err<=SPEC_ARCSEC).mean()*100),
                "oracle_MAE": float(np.min(np.abs(pred_mat[idx]-y[idx,None]), axis=1).mean()),
            })
    oracle_router_predictability_df = pd.DataFrame(oracle_router_rows)
    display(oracle_router_predictability_df)


In [ ]:
if "pred_mat" not in globals():
    print('Expert predictions are unavailable.')
else:
    disagreement = np.std(pred_mat, axis=1)
    prediction_map = {
        "expert_mean": np.mean(pred_mat, axis=1),
        "deterministic_elevation_interp": pred_det_interp,
    }
    if "moe_rx_predictions" in globals():
        for key, values in moe_rx_predictions.items():
            feature_set, model_name, split_name = key
            if split_name == "test":
                full = np.full(len(y), np.nan)
                full[test_idx] = values
                prediction_map[f"stack_{feature_set}_{model_name}"] = full

    coverage_levels = [.25, .50, .70, .80, .90, 1.0]
    selective_rows = []
    for model_name, prediction in prediction_map.items():
        valid_test = test_idx[np.isfinite(prediction[test_idx])]
        order = valid_test[np.argsort(disagreement[valid_test])]
        for coverage in coverage_levels:
            n_keep = max(1, int(np.ceil(len(order)*coverage)))
            keep = order[:n_keep]
            err = np.abs(y[keep]-prediction[keep])
            selective_rows.append({
                "model": model_name, "coverage_percent": 100*n_keep/len(order), "count": n_keep,
                "max_expert_std": float(np.max(disagreement[keep])), "MAE": float(err.mean()),
                "within_5_percent": float((err<=SPEC_ARCSEC).mean()*100), "p90_abs_error": float(np.quantile(err,.90)),
            })
    selective_disagreement_curve_df = pd.DataFrame(selective_rows)
    display(selective_disagreement_curve_df)

    # Select thresholds on validation, then apply them unchanged to the historical test.
    threshold_rows = []
    for desired_coverage in [.50, .70, .80, .90]:
        threshold = float(np.quantile(disagreement[val_idx], desired_coverage))
        for model_name, prediction in prediction_map.items():
            keep = test_idx[(disagreement[test_idx] <= threshold) & np.isfinite(prediction[test_idx])]
            if len(keep) == 0:
                continue
            err = np.abs(y[keep]-prediction[keep])
            threshold_rows.append({
                "model": model_name, "validation_target_coverage_percent": desired_coverage*100,
                "validation_expert_std_threshold": threshold, "test_coverage_percent": len(keep)/len(test_idx)*100,
                "test_count": len(keep), "test_MAE": float(err.mean()),
                "test_within_5_percent": float((err<=SPEC_ARCSEC).mean()*100), "test_p90_abs_error": float(np.quantile(err,.90)),
            })
    selective_disagreement_threshold_df = pd.DataFrame(threshold_rows)
    display(selective_disagreement_threshold_df)

    fig, ax = plt.subplots(figsize=(8,5))
    for model_name, group in selective_disagreement_curve_df.groupby("model", observed=True):
        ax.plot(group["coverage_percent"], group["MAE"], marker="o", label=model_name)
    ax.axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
    ax.set_xlabel('Retained coverage [%]')
    ax.set_ylabel("MAE [arcsec]")
    ax.set_title('Selective prediction using expert disagreement')
    ax.grid(True, alpha=.25)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
if "F_oracle" not in globals():
    F_oracle = fb.transform(X_seq, X_elev, meta).reset_index(drop=True)
if "F_oracle_rx" not in globals():
    F_oracle_rx = pd.concat([F_oracle, receiver_one_hot(meta)], axis=1)
quality_reference = meta.iloc[train_idx].copy()
quality_rows = []
for receiver_code, group in quality_reference.groupby("ReceiverCode", observed=True):
    hpbw = pd.to_numeric(group["LonFitHPBWArcsec"], errors="coerce").dropna()
    sigma = pd.to_numeric(group["FitXelSigmaArcsec"], errors="coerce").dropna()
    if len(group) < 10 or len(hpbw) < 10 or len(sigma) < 10:
        continue
    h_med = float(hpbw.median())
    h_mad = float(np.median(np.abs(hpbw-h_med))*1.4826)
    s_med = float(sigma.median())
    s_mad = float(np.median(np.abs(sigma-s_med))*1.4826)
    quality_rows.append({
        "ReceiverCode": receiver_code, "train_count": len(group),
        "HPBW_median": h_med, "HPBW_low": max(0.5*h_med, h_med-6*max(h_mad, 1.0)),
        "HPBW_high": min(1.5*h_med, h_med+6*max(h_mad, 1.0)),
        "sigma_median": s_med, "sigma_high": min(2.0, s_med+6*max(s_mad, .05)),
    })
quality_thresholds_df = pd.DataFrame(quality_rows)
display(quality_thresholds_df)

threshold_by_receiver = quality_thresholds_df.set_index("ReceiverCode") if len(quality_thresholds_df) else pd.DataFrame()
quality_mask = np.zeros(len(meta), dtype=bool)
quality_reason = np.full(len(meta), "receiver_not_calibrated", dtype=object)
for i, row in meta.reset_index(drop=True).iterrows():
    code = row.get("ReceiverCode")
    if code not in threshold_by_receiver.index:
        continue
    threshold = threshold_by_receiver.loc[code]
    hpbw = pd.to_numeric(pd.Series([row.get("LonFitHPBWArcsec")]), errors="coerce").iloc[0]
    sigma = pd.to_numeric(pd.Series([row.get("FitXelSigmaArcsec")]), errors="coerce").iloc[0]
    if not np.isfinite(hpbw) or not np.isfinite(sigma):
        quality_reason[i] = "missing_fit_quality"
    elif sigma > threshold["sigma_high"]:
        quality_reason[i] = "sigma_anomalous"
    elif not (threshold["HPBW_low"] <= hpbw <= threshold["HPBW_high"]):
        quality_reason[i] = "HPBW_anomalous"
    else:
        quality_mask[i] = True
        quality_reason[i] = "accepted"

quality_filter_summary_df = pd.DataFrame({"split": ["train"]*len(train_idx)+["val"]*len(val_idx)+["test"]*len(test_idx),
    "reason": np.r_[quality_reason[train_idx], quality_reason[val_idx], quality_reason[test_idx]]
}).groupby(["split","reason"], observed=True).size().rename("count").reset_index()
display(quality_filter_summary_df)

quality_train_idx = train_idx[quality_mask[train_idx]]
quality_val_idx = val_idx[quality_mask[val_idx]]
quality_test_idx = test_idx[quality_mask[test_idx]]
quality_baseline_rows = []
for training_set, tr in [("all_training", train_idx), ("quality_training", quality_train_idx)]:
    model = clone(models["huber_physical"]).fit(F_oracle_rx.iloc[tr], y[tr])
    for eval_set, ev in [("quality_val_common", quality_val_idx), ("quality_test_common", quality_test_idx)]:
        if len(ev) == 0:
            continue
        pred = model.predict(F_oracle_rx.iloc[ev])
        err = np.abs(y[ev]-pred)
        quality_baseline_rows.append({
            "training_set": training_set, "eval_set": eval_set, "n_train": len(tr), "n_eval": len(ev),
            "MAE": float(err.mean()), "within_5_percent": float((err<=SPEC_ARCSEC).mean()*100),
            "p90_abs_error": float(np.quantile(err,.90)),
        })
quality_baseline_df = pd.DataFrame(quality_baseline_rows)
display(quality_baseline_df)

quality_meta = meta.iloc[np.flatnonzero(quality_mask)].copy()
quality_repeat_pairs_df = build_repeatability_pairs(quality_meta, X_seq)
quality_repeatability_df = pd.DataFrame([{
    "criterion": "quality_broad_6h_el3", "pair_count": len(quality_repeat_pairs_df),
    "median_abs_delta_target": float(quality_repeat_pairs_df["abs_dtarget"].median()) if len(quality_repeat_pairs_df) else np.nan,
    "p90_abs_delta_target": float(quality_repeat_pairs_df["abs_dtarget"].quantile(.90)) if len(quality_repeat_pairs_df) else np.nan,
}, {
    "criterion": "quality_same_rx_60m_el1",
    "pair_count": int(((quality_repeat_pairs_df["dt_min"]<=60)&(quality_repeat_pairs_df["delev_deg"]<=1)&quality_repeat_pairs_df["same_receiver_code"]).sum()),
    "median_abs_delta_target": float(quality_repeat_pairs_df.loc[
        (quality_repeat_pairs_df["dt_min"]<=60)&(quality_repeat_pairs_df["delev_deg"]<=1)&quality_repeat_pairs_df["same_receiver_code"], "abs_dtarget"
    ].median()),
    "p90_abs_delta_target": float(quality_repeat_pairs_df.loc[
        (quality_repeat_pairs_df["dt_min"]<=60)&(quality_repeat_pairs_df["delev_deg"]<=1)&quality_repeat_pairs_df["same_receiver_code"], "abs_dtarget"
    ].quantile(.90)),
}])
display(quality_repeatability_df)


## Ramo EL: audit, modellazione e verifica della specifica


In [ ]:
POINTING_AUDIT_CANDIDATES = ["control_crosscheck_pointing_audit.csv"]
pointing_audit_path = find_data_file(POINTING_AUDIT_CANDIDATES)
pointing_audit = pd.read_csv(pointing_audit_path)
pointing_audit["Time"] = pd.to_datetime(pointing_audit["Time"], errors="coerce")
pointing_audit = pointing_audit.dropna(subset=["Time"]).sort_values("Time").drop_duplicates("Time", keep="last")
for col in [
    "Offset_EL_arcsec", "dat_el_delta_arcsec", "dat_time_delta_seconds", "dat_el_sigma_arcsec",
    "dat_latfit_hpbw_arcsec", "azel_nonzero_count_since_zero", "azel_latest_el_deg",
    "minutes_since_complete_offset_reset", "minutes_since_nonzero_offset_command",
]:
    if col in pointing_audit:
        pointing_audit[col] = pd.to_numeric(pointing_audit[col], errors="coerce")

pa_by_time = pointing_audit.set_index("Time")
embedded_el = real_raw.loc[real_raw["El Offset"].notna(), ["Timestamp", "El Offset"]].copy()
embedded_el["audit_EL"] = embedded_el["Timestamp"].map(pa_by_time["Offset_EL_arcsec"])
embedded_el["embedded_minus_audit_EL"] = embedded_el["El Offset"] - embedded_el["audit_EL"]
embedded_el["dat_EL_delta"] = embedded_el["Timestamp"].map(pa_by_time["dat_el_delta_arcsec"])

el_target_audit_df = pd.DataFrame([{
    "embedded_EL_targets": len(embedded_el),
    "matched_to_control_audit": int(embedded_el["audit_EL"].notna().sum()),
    "max_abs_embedded_minus_audit_arcsec": float(embedded_el["embedded_minus_audit_EL"].abs().max()),
    "median_abs_embedded_minus_audit_arcsec": float(embedded_el["embedded_minus_audit_EL"].abs().median()),
    "dat_matches": int(embedded_el["dat_EL_delta"].notna().sum()),
    "max_abs_audit_minus_dat_arcsec": float(embedded_el["dat_EL_delta"].abs().max()),
    "active_azel_offsets_excluded": int(active_offset_mask.sum()),
}])
display(el_target_audit_df)

el_meta = meta.copy().reset_index(drop=True)
el_meta["Target"] = el_meta["Timestamp"].map(pa_by_time["Offset_EL_arcsec"])
if el_meta["Target"].isna().any():
    raise ValueError(f"Missing EL targets among clean XEL windows: {int(el_meta['Target'].isna().sum())}")
y_el = el_meta["Target"].to_numpy(dtype=float)
for source_col, dest_col in [
    ("dat_el_sigma_arcsec", "FitElSigmaArcsec"),
    ("dat_latfit_hpbw_arcsec", "LatFitHPBWArcsec"),
    ("minutes_since_complete_offset_reset", "MinutesSinceCompleteReset"),
    ("minutes_since_nonzero_offset_command", "MinutesSinceNonzeroOffset"),
    ("azel_latest_el_deg", "LatestElOffsetDeg"),
    ("azel_nonzero_count_since_zero", "ActiveAzelOffsetCountAudit"),
]:
    if source_col in pa_by_time.columns:
        el_meta[dest_col] = el_meta["Timestamp"].map(pa_by_time[source_col])

el_target_distribution_df = pd.DataFrame([{
    "count": len(y_el), "mean": float(np.mean(y_el)), "std": float(np.std(y_el)),
    "median_abs": float(np.median(np.abs(y_el))), "p90_abs": float(np.quantile(np.abs(y_el), .90)),
    "gt15_percent": float((np.abs(y_el)>15).mean()*100), "min": float(np.min(y_el)), "max": float(np.max(y_el)),
}])
display(el_target_distribution_df)


In [ ]:
el_repeatability_pairs_df = build_repeatability_pairs(el_meta, X_seq, fit_sigma_col="FitElSigmaArcsec")
el_repeatability_rows = []
for name, max_dt, max_el, max_az, same_rx, max_mean_t, max_pattern_t in repeatability_criteria:
    mask = (el_repeatability_pairs_df["dt_min"] <= max_dt) & (el_repeatability_pairs_df["delev_deg"] <= max_el)
    if max_az is not None:
        mask &= el_repeatability_pairs_df["daz_deg"] <= max_az
    if same_rx:
        mask &= el_repeatability_pairs_df["same_receiver_code"]
    if max_mean_t is not None:
        mask &= el_repeatability_pairs_df["thermal_mean_delta_C"] <= max_mean_t
    if max_pattern_t is not None:
        mask &= el_repeatability_pairs_df["thermal_centered_rmse_C"] <= max_pattern_t
    sub = el_repeatability_pairs_df.loc[mask]
    med = float(sub["abs_dtarget"].median()) if len(sub) else np.nan
    ci_low, ci_high = bootstrap_quantile_ci(sub["abs_dtarget"], random_state=RANDOM_SEED+100+len(el_repeatability_rows))
    el_repeatability_rows.append({
        "criterion": name, "pair_count": len(sub), "median_abs_delta_EL": med,
        "median_ci95_low": ci_low, "median_ci95_high": ci_high,
        "p75_abs_delta_EL": float(sub["abs_dtarget"].quantile(.75)) if len(sub) else np.nan,
        "p90_abs_delta_EL": float(sub["abs_dtarget"].quantile(.90)) if len(sub) else np.nan,
        "gaussian_equiv_single_measurement_MAE": med/(norm.ppf(.75)*np.sqrt(np.pi)) if np.isfinite(med) else np.nan,
        "median_fit_pair_sigma": float(sub["fit_pair_sigma_arcsec"].median()) if len(sub) else np.nan,
    })
el_repeatability_df = pd.DataFrame(el_repeatability_rows)
display(el_repeatability_df)


In [ ]:
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge

if "F_oracle" not in globals():
    F_oracle = fb.transform(X_seq, X_elev, meta).reset_index(drop=True)
R_el = receiver_one_hot(meta)

el_spline = SplineTransformer(n_knots=7, degree=3, include_bias=False, extrapolation="linear")
az_spline = SplineTransformer(n_knots=9, degree=3, include_bias=False, extrapolation="periodic")
el_spline.fit(el_meta.loc[train_idx, ["Elevation"]])
az_spline.fit(el_meta.loc[train_idx, ["Azimuth"]])
G_el = pd.DataFrame(el_spline.transform(el_meta[["Elevation"]]), columns=[f"el_spline_{i}" for i in range(el_spline.n_features_out_)])
G_az = pd.DataFrame(az_spline.transform(el_meta[["Azimuth"]]), columns=[f"az_spline_{i}" for i in range(az_spline.n_features_out_)])
geometry_el = pd.concat([G_el, G_az, R_el], axis=1)

solar_el = solar_all.copy().reset_index(drop=True)
relative_sun = np.deg2rad((el_meta["Azimuth"]-solar_el["solar_azimuth"]+180)%360-180)
solar_features_el = pd.DataFrame({
    "sun_up": (solar_el["solar_elevation"]>0).astype(float),
    "solar_height": np.maximum(np.sin(np.deg2rad(solar_el["solar_elevation"])), 0),
    "solar_az_sin": np.sin(np.deg2rad(solar_el["solar_azimuth"])),
    "solar_az_cos": np.cos(np.deg2rad(solar_el["solar_azimuth"])),
    "relative_sun_sin": np.sin(relative_sun), "relative_sun_cos": np.cos(relative_sun),
})

control_features_el = pd.DataFrame({
    "minutes_since_complete_reset": el_meta.get("MinutesSinceCompleteReset", pd.Series(np.nan, index=el_meta.index)),
    "minutes_since_nonzero_offset": el_meta.get("MinutesSinceNonzeroOffset", pd.Series(np.nan, index=el_meta.index)),
    "latest_el_offset_deg": el_meta.get("LatestElOffsetDeg", pd.Series(np.nan, index=el_meta.index)),
})

thermal_el = F_oracle.drop(columns=[c for c in F_oracle.columns if c in ["elevation", "sin_el", "cos_el", "az_sin", "az_cos"]], errors="ignore")
interaction_source = [c for c in thermal_el.columns if c.startswith("pc_")][:16]
interactions_el = pd.DataFrame(index=thermal_el.index)
sin_el_all = np.sin(np.deg2rad(el_meta["Elevation"].to_numpy(dtype=float)))
cos_el_all = np.cos(np.deg2rad(el_meta["Elevation"].to_numpy(dtype=float)))
for col in interaction_source:
    interactions_el[f"{col}_x_sin_el"] = thermal_el[col].to_numpy()*sin_el_all
    interactions_el[f"{col}_x_cos_el"] = thermal_el[col].to_numpy()*cos_el_all

el_feature_sets = {
    "geometry_only": geometry_el,
    "thermal_only": pd.concat([F_oracle, R_el], axis=1),
    "geometry_plus_thermal": pd.concat([geometry_el, thermal_el], axis=1),
    "geometry_thermal_interactions": pd.concat([geometry_el, thermal_el, interactions_el], axis=1),
    "geometry_thermal_solar": pd.concat([geometry_el, thermal_el, solar_features_el], axis=1),
    "geometry_thermal_control": pd.concat([geometry_el, thermal_el, control_features_el], axis=1),
}
el_candidate_models = {
    "ridge": models["ridge_physical"],
    "huber": models["huber_physical"],
    "hist_gradient_boosting": models["hist_gradient_boosting"],
}
el_model_rows, el_model_predictions, el_fitted_models = [], {}, {}
for feature_set, Xuse in el_feature_sets.items():
    for model_name, estimator in el_candidate_models.items():
        fitted = clone(estimator).fit(Xuse.iloc[train_idx], y_el[train_idx])
        el_fitted_models[(feature_set, model_name)] = fitted
        for split_name, idx in [("val", val_idx), ("test", test_idx)]:
            pred = fitted.predict(Xuse.iloc[idx])
            el_model_predictions[(feature_set, model_name, split_name)] = pred
            err = np.abs(y_el[idx]-pred)
            el_model_rows.append({
                "feature_set": feature_set, "model": model_name, "split": split_name, "count": len(idx),
                "MAE": float(err.mean()), "RMSE": float(np.sqrt(np.mean((y_el[idx]-pred)**2))),
                "R2": float(r2_score(y_el[idx], pred)), "within_5_percent": float((err<=SPEC_ARCSEC).mean()*100),
                "p90_abs_error": float(np.quantile(err,.90)),
            })
el_model_benchmark_df = pd.DataFrame(el_model_rows).sort_values(["split", "MAE"])
display(el_model_benchmark_df)

best_el_val = el_model_benchmark_df[el_model_benchmark_df["split"]=="val"].iloc[0]
BEST_EL_FEATURE_SET, BEST_EL_MODEL = best_el_val["feature_set"], best_el_val["model"]
best_el_test_row = el_model_benchmark_df[
    (el_model_benchmark_df["split"]=="test") &
    (el_model_benchmark_df["feature_set"]==BEST_EL_FEATURE_SET) &
    (el_model_benchmark_df["model"]==BEST_EL_MODEL)
]
el_selected_model_df = pd.DataFrame([{
    "selected_feature_set": BEST_EL_FEATURE_SET, "selected_model": BEST_EL_MODEL,
    "validation_MAE": float(best_el_val["MAE"]), "test_MAE": float(best_el_test_row.iloc[0]["MAE"]),
    "test_within_5_percent": float(best_el_test_row.iloc[0]["within_5_percent"]),
    "test_p90_abs_error": float(best_el_test_row.iloc[0]["p90_abs_error"]),
}])
display(el_selected_model_df)


X_weighted = el_feature_sets["geometry_thermal_interactions"].copy()
imputer_w = SimpleImputer(strategy="median")
scaler_w = StandardScaler()
Xtr_w = scaler_w.fit_transform(imputer_w.fit_transform(X_weighted.iloc[train_idx]))
Xv_w = scaler_w.transform(imputer_w.transform(X_weighted.iloc[val_idx]))
Xte_w = scaler_w.transform(imputer_w.transform(X_weighted.iloc[test_idx]))
sigma_train = pd.to_numeric(el_meta.iloc[train_idx]["FitElSigmaArcsec"], errors="coerce").to_numpy(dtype=float)
sigma_fill = np.nanmedian(sigma_train)
sigma_train = np.where(np.isfinite(sigma_train), sigma_train, sigma_fill)
weight_rows = []
for floor in [.25, .5, 1.0, 2.0]:
    weights = 1/(sigma_train**2+floor**2)
    weights = np.clip(weights/np.median(weights), .2, 5.0)
    for alpha in [.1, 1.0, 10.0, 100.0]:
        mdl = Ridge(alpha=alpha).fit(Xtr_w, y_el[train_idx], sample_weight=weights)
        for split_name, yt, Xev in [("val", y_el[val_idx], Xv_w), ("test", y_el[test_idx], Xte_w)]:
            pred = mdl.predict(Xev)
            weight_rows.append({"sigma_floor": floor, "alpha": alpha, "split": split_name, "MAE": mean_absolute_error(yt,pred)})
el_weighted_ridge_df = pd.DataFrame(weight_rows)
display(el_weighted_ridge_df.sort_values(["split","MAE"]))


In [ ]:
el_target_table = real_raw.loc[real_raw["El Offset"].notna() & ~active_offset_mask, [
    "Timestamp", "SRT Elevation Ange[DEG]", "Azimuth", "El Offset"
]].copy()
history_rows, history_coverage_rows, history_prediction_tables = [], [], {}
val_boundary = meta.iloc[val_idx]["Timestamp"].min()
test_boundary = meta.iloc[test_idx]["Timestamp"].min()

for hours, seq_len in [(2,60), (4,120), (8,240)]:
    Xh, Eh, yh, mh, skip_h = build_supervised_windows(
        real_raw, sensor_cols, target_col="El Offset", target_table=el_target_table, seq_len=seq_len
    )
    mh = mh.merge(true_xel_audit[["Timestamp","receiver_class","receiver_band"]], on="Timestamp", how="left")
    mh = mh.rename(columns={"receiver_class":"ReceiverCode", "receiver_band":"ReceiverBand"})
    trh = np.flatnonzero(mh["Timestamp"] < val_boundary)
    vah = np.flatnonzero((mh["Timestamp"] >= val_boundary) & (mh["Timestamp"] < test_boundary))
    teh = np.flatnonzero(mh["Timestamp"] >= test_boundary)
    history_coverage_rows.append({
        "window_hours": hours, "valid_windows": len(mh), "n_train":len(trh), "n_val":len(vah), "n_test":len(teh),
        **{f"skip_{k}":v for k,v in skip_h.items()},
    })
    if min(len(trh),len(vah),len(teh)) < 10:
        continue
    fbh = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(Xh[trh], Eh[trh])
    Fh = fbh.transform(Xh, Eh, mh).reset_index(drop=True)
    Fh = pd.concat([Fh, receiver_one_hot(mh)], axis=1)
    candidate_preds = {}
    for model_name in ["ridge_physical", "huber_physical"]:
        fitted = clone(models[model_name]).fit(Fh.iloc[trh], yh[trh])
        pv, pt = fitted.predict(Fh.iloc[vah]), fitted.predict(Fh.iloc[teh])
        candidate_preds[model_name] = (pv,pt)
        for split_name, idx, pred in [("val",vah,pv),("test",teh,pt)]:
            err = np.abs(yh[idx]-pred)
            history_rows.append({
                "window_hours":hours,"model":model_name,"split":split_name,"count":len(idx),
                "MAE":float(err.mean()),"within_5_percent":float((err<=SPEC_ARCSEC).mean()*100),
                "p90_abs_error":float(np.quantile(err,.90)),
            })
    val_maes = {name:mean_absolute_error(yh[vah],preds[0]) for name,preds in candidate_preds.items()}
    chosen = min(val_maes,key=val_maes.get)
    history_prediction_tables[hours] = pd.DataFrame({"Timestamp":mh.iloc[teh]["Timestamp"].to_numpy(),"y_true":yh[teh],"y_pred":candidate_preds[chosen][1],"model":chosen})
    del Xh, Eh, yh, Fh

el_history_coverage_df = pd.DataFrame(history_coverage_rows)
el_history_benchmark_df = pd.DataFrame(history_rows).sort_values(["split","MAE"])
display(el_history_coverage_df)
display(el_history_benchmark_df)

common_times = None
for table in history_prediction_tables.values():
    times = set(table["Timestamp"])
    common_times = times if common_times is None else common_times & times
common_rows = []
for hours, table in history_prediction_tables.items():
    sub = table[table["Timestamp"].isin(common_times)]
    err = np.abs(sub["y_true"]-sub["y_pred"])
    common_rows.append({"window_hours":hours,"selected_model":sub["model"].iloc[0] if len(sub) else None,
                        "common_test_count":len(sub),"MAE":float(err.mean()) if len(sub) else np.nan,
                        "within_5_percent":float((err<=SPEC_ARCSEC).mean()*100) if len(sub) else np.nan,
                        "p90_abs_error":float(np.quantile(err,.90)) if len(sub) else np.nan})
el_history_common_test_df = pd.DataFrame(common_rows)
display(el_history_common_test_df)


In [ ]:
best_el_test_pred = el_model_predictions[(BEST_EL_FEATURE_SET,BEST_EL_MODEL,"test")]
el_test_diag = el_meta.iloc[test_idx].copy().reset_index(drop=True)
el_test_diag["y_true"] = y_el[test_idx]
el_test_diag["y_pred"] = best_el_test_pred
el_test_diag["residual"] = el_test_diag["y_true"]-el_test_diag["y_pred"]
el_test_diag["abs_error"] = el_test_diag["residual"].abs()
el_test_diag["elev_band"] = pd.cut(el_test_diag["Elevation"],[0,20,30,45,60,75,90],include_lowest=True)
el_test_diag["az_band"] = pd.cut(el_test_diag["Azimuth"],np.arange(0,361,45),include_lowest=True,right=False)
el_test_diag["sun_up"] = solar_all.iloc[test_idx]["solar_elevation"].to_numpy()>0
el_test_diag["thermal_level"] = pd.cut(np.mean(X_seq[test_idx,-1,:],axis=1),[-np.inf,15,25,np.inf])

el_residual_by_elevation_df = el_test_diag.groupby("elev_band",observed=True).agg(
    count=("abs_error","size"),bias=("residual","mean"),MAE=("abs_error","mean"),
    within_5_percent=("abs_error",lambda s:(s<=SPEC_ARCSEC).mean()*100),p90_abs_error=("abs_error",lambda s:s.quantile(.90)),
).reset_index()
el_residual_by_azimuth_df = el_test_diag.groupby("az_band",observed=True).agg(
    count=("abs_error","size"),bias=("residual","mean"),MAE=("abs_error","mean"),p90_abs_error=("abs_error",lambda s:s.quantile(.90)),
).reset_index()
el_residual_by_receiver_df = el_test_diag.groupby("ReceiverBand",observed=True).agg(
    count=("abs_error","size"),bias=("residual","mean"),MAE=("abs_error","mean"),
    median_fit_sigma=("FitElSigmaArcsec","median"),median_HPBW=("LatFitHPBWArcsec","median"),
).reset_index()
el_residual_by_sun_thermal_df = el_test_diag.groupby(["sun_up","thermal_level"],observed=True).agg(
    count=("abs_error","size"),bias=("residual","mean"),MAE=("abs_error","mean"),p90_abs_error=("abs_error",lambda s:s.quantile(.90)),
).reset_index()
display(el_residual_by_elevation_df)
display(el_residual_by_azimuth_df)
display(el_residual_by_receiver_df)
display(el_residual_by_sun_thermal_df)

# Estimate EL quality thresholds on training data separately for each receiver.
el_quality_rows=[]
for code,g in el_meta.iloc[train_idx].groupby("ReceiverCode",observed=True):
    hp=pd.to_numeric(g["LatFitHPBWArcsec"],errors="coerce").dropna(); sg=pd.to_numeric(g["FitElSigmaArcsec"],errors="coerce").dropna()
    if len(g)<10 or len(hp)<10 or len(sg)<10: continue
    hm=float(hp.median()); hmad=float(np.median(np.abs(hp-hm))*1.4826); sm=float(sg.median()); smad=float(np.median(np.abs(sg-sm))*1.4826)
    el_quality_rows.append({"ReceiverCode":code,"train_count":len(g),"HPBW_median":hm,
                            "HPBW_low":max(.5*hm,hm-6*max(hmad,1)),"HPBW_high":min(1.5*hm,hm+6*max(hmad,1)),
                            "sigma_median":sm,"sigma_high":min(2.0,sm+6*max(smad,.05))})
el_quality_thresholds_df=pd.DataFrame(el_quality_rows)
thr=el_quality_thresholds_df.set_index("ReceiverCode")
el_quality_mask=np.zeros(len(el_meta),dtype=bool); reasons=np.full(len(el_meta),"receiver_not_calibrated",dtype=object)
for i,row in el_meta.iterrows():
    code=row["ReceiverCode"]
    if code not in thr.index: continue
    t=thr.loc[code]; hp=row["LatFitHPBWArcsec"]; sg=row["FitElSigmaArcsec"]
    if not np.isfinite(hp) or not np.isfinite(sg): reasons[i]="missing_fit_quality"
    elif sg>t["sigma_high"]: reasons[i]="sigma_anomalous"
    elif not(t["HPBW_low"]<=hp<=t["HPBW_high"]): reasons[i]="HPBW_anomalous"
    else: el_quality_mask[i]=True; reasons[i]="accepted"
el_quality_filter_summary_df=pd.DataFrame({"split":["train"]*len(train_idx)+["val"]*len(val_idx)+["test"]*len(test_idx),
    "reason":np.r_[reasons[train_idx],reasons[val_idx],reasons[test_idx]]}).groupby(["split","reason"]).size().rename("count").reset_index()
display(el_quality_thresholds_df)
display(el_quality_filter_summary_df)

X_best_el=el_feature_sets[BEST_EL_FEATURE_SET]
qtr=train_idx[el_quality_mask[train_idx]]; qva=val_idx[el_quality_mask[val_idx]]; qte=test_idx[el_quality_mask[test_idx]]
el_quality_benchmark_rows=[]
for training_set,tr in [("all_training",train_idx),("quality_training",qtr)]:
    mdl=clone(el_candidate_models[BEST_EL_MODEL]).fit(X_best_el.iloc[tr],y_el[tr])
    for eval_set,ev in [("quality_val_common",qva),("quality_test_common",qte)]:
        if len(ev)==0: continue
        pred=mdl.predict(X_best_el.iloc[ev]); err=np.abs(y_el[ev]-pred)
        el_quality_benchmark_rows.append({"training_set":training_set,"eval_set":eval_set,"n_train":len(tr),"n_eval":len(ev),
                                          "MAE":float(err.mean()),"within_5_percent":float((err<=5).mean()*100),"p90_abs_error":float(np.quantile(err,.90))})
el_quality_benchmark_df=pd.DataFrame(el_quality_benchmark_rows)
display(el_quality_benchmark_df)


In [ ]:
# Measured XEL is diagnostic only; operational features use predicted XEL.

from scipy.stats import spearmanr

def _fold_features(tr, ev):
    builder = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq[tr], X_elev[tr])
    ftr = builder.transform(X_seq[tr], X_elev[tr], meta.iloc[tr]).reset_index(drop=True)
    fev = builder.transform(X_seq[ev], X_elev[ev], meta.iloc[ev]).reset_index(drop=True)
    r_all = receiver_one_hot(meta)
    return (
        pd.concat([ftr, r_all.iloc[tr].reset_index(drop=True)], axis=1),
        pd.concat([fev, r_all.iloc[ev].reset_index(drop=True)], axis=1),
    )

def _inner_oof_xel(outer_train, n_splits=3):
    pred = np.full(len(outer_train), np.nan)
    splitter = TimeSeriesSplit(n_splits=n_splits)
    for inner_tr_local, inner_ev_local in splitter.split(outer_train):
        inner_tr = outer_train[inner_tr_local]
        inner_ev = outer_train[inner_ev_local]
        fit_tr, fit_ev = _fold_features(inner_tr, inner_ev)
        mdl = clone(models["huber_physical"]).fit(fit_tr, y[inner_tr])
        pred[inner_ev_local] = mdl.predict(fit_ev)
    return pred

coupling_rows, coupling_prediction_rows = [], []
for fold, (tr_local, ev_local) in enumerate(TimeSeriesSplit(n_splits=4).split(pretest_idx), 1):
    tr, ev = pretest_idx[tr_local], pretest_idx[ev_local]
    Xtr_c, Xev_c = _fold_features(tr, ev)

    xel_model = clone(models["huber_physical"]).fit(Xtr_c, y[tr])
    el_model = clone(models["huber_physical"]).fit(Xtr_c, y_el[tr])
    pred_xel_ev = xel_model.predict(Xev_c)
    pred_el_base = el_model.predict(Xev_c)
    residual_xel = y[ev] - pred_xel_ev
    residual_el = y_el[ev] - pred_el_base
    coupling_rows.append({
        "fold": fold, "count": len(ev),
        "target_pearson": float(np.corrcoef(y[ev], y_el[ev])[0,1]),
        "target_spearman": float(spearmanr(y[ev], y_el[ev]).statistic),
        "residual_pearson": float(np.corrcoef(residual_xel, residual_el)[0,1]),
        "residual_spearman": float(spearmanr(residual_xel, residual_el).statistic),
    })

    pred_xel_oof_tr = _inner_oof_xel(tr)
    valid = np.isfinite(pred_xel_oof_tr)
    for xel_source, train_xel_feature, eval_xel_feature, operational in [
        ("without_XEL", None, None, True),
        ("predicted_XEL_out_of_fold", pred_xel_oof_tr, pred_xel_ev, True),
        ("measured_XEL_diagnostic_only", y[tr], y[ev], False),
    ]:
        # Use the same training subset for all variants.
        train_mask = valid
        if xel_source == "without_XEL":
            Xtr_stack = Xtr_c.iloc[train_mask].copy()
            Xev_stack = Xev_c.copy()
        else:
            Xtr_stack = Xtr_c.iloc[train_mask].copy()
            Xev_stack = Xev_c.copy()
            Xtr_stack["xel_auxiliary"] = np.asarray(train_xel_feature)[train_mask]
            Xev_stack["xel_auxiliary"] = np.asarray(eval_xel_feature)
        for model_name in ["ridge_physical", "huber_physical"]:
            mdl = clone(models[model_name]).fit(Xtr_stack, y_el[tr][train_mask])
            pred = mdl.predict(Xev_stack)
            err = np.abs(y_el[ev]-pred)
            coupling_prediction_rows.append({
                "fold": fold, "xel_source": xel_source, "operational": operational,
                "model": model_name.replace("_physical", ""), "n_train": int(train_mask.sum()),
                "n_eval": len(ev), "MAE": float(err.mean()),
                "within_5_percent": float((err<=SPEC_ARCSEC).mean()*100),
                "p90_abs_error": float(np.quantile(err,.90)),
            })

el_xel_residual_correlation_df = pd.DataFrame(coupling_rows)
el_xel_stacking_cv_df = pd.DataFrame(coupling_prediction_rows)
el_xel_stacking_summary_df = el_xel_stacking_cv_df.groupby(
    ["xel_source","operational","model"], observed=True
).agg(folds=("fold","nunique"), MAE_mean=("MAE","mean"), MAE_std=("MAE","std"),
      within_5_mean=("within_5_percent","mean"), p90_mean=("p90_abs_error","mean")).reset_index().sort_values("MAE_mean")
display(el_xel_residual_correlation_df)
display(el_xel_stacking_summary_df)


In [ ]:
class MultiScaleFeatureBuilder:
    """Fit training-only PCA for state, centered patterns and 30/60/120-minute trends."""
    def __init__(self, n_components=6):
        self.n_components = n_components
        self.views = ["current", "centered", "trend_30m", "trend_60m", "trend_120m"]
        self.scalers = {name: StandardScaler() for name in self.views}
        self.pcas = {name: PCA(n_components=n_components, random_state=RANDOM_SEED) for name in self.views}

    def _views(self, X):
        cur = X[:,-1,:]
        lag30 = X[:,max(0,X.shape[1]-16),:]
        lag60 = X[:,max(0,X.shape[1]-31),:]
        return {
            "current": cur,
            "centered": cur-cur.mean(axis=1,keepdims=True),
            "trend_30m": cur-lag30,
            "trend_60m": cur-lag60,
            "trend_120m": cur-X[:,0,:],
        }

    def fit(self, X):
        for name, values in self._views(X).items():
            self.pcas[name].fit(self.scalers[name].fit_transform(values))
        return self

    def transform(self, X, elev, meta_df):
        out = scalar_physical_features(X,elev).reset_index(drop=True)
        out = pd.concat([out,temporal_context_features(meta_df)],axis=1)
        for name, values in self._views(X).items():
            z = self.pcas[name].transform(self.scalers[name].transform(values))
            for j in range(z.shape[1]):
                out[f"ms_{name}_{j+1:02d}"] = z[:,j]

        cur=X[:,-1,:]; recent=X[:,-16:,:]
        out["current_iqr"] = np.quantile(cur,.75,axis=1)-np.quantile(cur,.25,axis=1)
        out["current_p90_p10"] = np.quantile(cur,.90,axis=1)-np.quantile(cur,.10,axis=1)
        out["recent_mean_change"] = cur.mean(axis=1)-recent[:,0,:].mean(axis=1)
        out["recent_pattern_change"] = np.std(cur-recent[:,0,:],axis=1)
        return out

multiscale_rows=[]
for fold,(tr_local,ev_local) in enumerate(TimeSeriesSplit(n_splits=4).split(pretest_idx),1):
    tr,ev=pretest_idx[tr_local],pretest_idx[ev_local]
    fb_base=FeatureBuilder(8,8).fit(X_seq[tr],X_elev[tr])
    base_tr=fb_base.transform(X_seq[tr],X_elev[tr],meta.iloc[tr]).reset_index(drop=True)
    base_ev=fb_base.transform(X_seq[ev],X_elev[ev],meta.iloc[ev]).reset_index(drop=True)
    ms=MultiScaleFeatureBuilder(6).fit(X_seq[tr])
    ms_tr=ms.transform(X_seq[tr],X_elev[tr],meta.iloc[tr]).reset_index(drop=True)
    ms_ev=ms.transform(X_seq[ev],X_elev[ev],meta.iloc[ev]).reset_index(drop=True)
    rx_tr=receiver_one_hot(meta.iloc[tr]); rx_ev=receiver_one_hot(meta.iloc[ev])
    feature_variants={
        "baseline_thermal":(pd.concat([base_tr,rx_tr],axis=1),pd.concat([base_ev,rx_ev],axis=1)),
        "multiscale_thermal":(pd.concat([ms_tr,rx_tr],axis=1),pd.concat([ms_ev,rx_ev],axis=1)),
    }
    if "pred_mat" in globals() and len(pred_mat)==len(y):
        fem_cols=[f"fem_expert_{e}" for e in MOE_EXPERT_ELEVS]
        fem_tr=pd.DataFrame(pred_mat[tr],columns=fem_cols)
        fem_ev=pd.DataFrame(pred_mat[ev],columns=fem_cols)
        fem_tr["fem_expert_std"]=np.std(pred_mat[tr],axis=1); fem_ev["fem_expert_std"]=np.std(pred_mat[ev],axis=1)
        feature_variants["baseline_plus_FEM_experts"]=(pd.concat([base_tr,rx_tr,fem_tr],axis=1),pd.concat([base_ev,rx_ev,fem_ev],axis=1))
    for feature_set,(xtr,xev) in feature_variants.items():
        for model_name in ["ridge_physical","huber_physical"]:
            mdl=clone(models[model_name]).fit(xtr,y_el[tr]); pred=mdl.predict(xev); err=np.abs(y_el[ev]-pred)
            multiscale_rows.append({"fold":fold,"feature_set":feature_set,"model":model_name.replace("_physical",""),
                                    "count":len(ev),"MAE":float(err.mean()),"within_5_percent":float((err<=5).mean()*100),
                                    "p90_abs_error":float(np.quantile(err,.90))})
el_multiscale_cv_df=pd.DataFrame(multiscale_rows)
el_multiscale_summary_df=el_multiscale_cv_df.groupby(["feature_set","model"],observed=True).agg(
    folds=("fold","nunique"),MAE_mean=("MAE","mean"),MAE_std=("MAE","std"),
    within_5_mean=("within_5_percent","mean"),p90_mean=("p90_abs_error","mean")).reset_index().sort_values("MAE_mean")
display(el_multiscale_summary_df)


In [ ]:
def _fit_weighted_pipeline(estimator, X, target, weights):
    fitted=clone(estimator)
    if isinstance(fitted,Pipeline):
        fitted.fit(X,target,model__sample_weight=weights)
    else:
        fitted.fit(X,target,sample_weight=weights)
    return fitted

def _specialist_prediction(estimator,Xtr,Xev,tr,ev,smooth=True):
    elev_tr=el_meta.iloc[tr]["Elevation"].to_numpy(); elev_ev=el_meta.iloc[ev]["Elevation"].to_numpy()
    low_mask=elev_tr<=65 if smooth else elev_tr<60
    high_mask=elev_tr>=55 if smooth else elev_tr>=60
    low=clone(estimator).fit(Xtr.iloc[low_mask],y_el[tr][low_mask])
    high=clone(estimator).fit(Xtr.iloc[high_mask],y_el[tr][high_mask])
    plow,phigh=low.predict(Xev),high.predict(Xev)
    if smooth:
        weight=np.clip((elev_ev-55)/10,0,1)
        return (1-weight)*plow+weight*phigh
    return np.where(elev_ev<60,plow,phigh)

specialist_rows=[]
for fold,(tr_local,ev_local) in enumerate(TimeSeriesSplit(n_splits=4).split(pretest_idx),1):
    tr,ev=pretest_idx[tr_local],pretest_idx[ev_local]
    Xtr_s,Xev_s=_fold_features(tr,ev)
    for model_name in ["ridge_physical","huber_physical"]:
        estimator=models[model_name]
        candidates={
            "global":clone(estimator).fit(Xtr_s,y_el[tr]).predict(Xev_s),
            "hard_experts_60":_specialist_prediction(estimator,Xtr_s,Xev_s,tr,ev,smooth=False),
            "smooth_experts_55_65":_specialist_prediction(estimator,Xtr_s,Xev_s,tr,ev,smooth=True),
        }
        for high_weight in [1.5,2.0,3.0]:
            weights=np.where(el_meta.iloc[tr]["Elevation"].to_numpy()>=60,high_weight,1.0)
            candidates[f"global_high_weight_{high_weight:g}"]=_fit_weighted_pipeline(estimator,Xtr_s,y_el[tr],weights).predict(Xev_s)
        for strategy,pred in candidates.items():
            err=np.abs(y_el[ev]-pred); high=el_meta.iloc[ev]["Elevation"].to_numpy()>=60
            specialist_rows.append({"fold":fold,"model":model_name.replace("_physical",""),"strategy":strategy,
                "count":len(ev),"high_count":int(high.sum()),"MAE":float(err.mean()),
                "MAE_below_60":float(err[~high].mean()) if (~high).any() else np.nan,
                "MAE_at_least_60":float(err[high].mean()) if high.any() else np.nan,
                "within_5_percent":float((err<=5).mean()*100),"p90_abs_error":float(np.quantile(err,.90))})
el_specialist_cv_df=pd.DataFrame(specialist_rows)
el_specialist_summary_df=el_specialist_cv_df.groupby(["model","strategy"],observed=True).agg(
    folds=("fold","nunique"),MAE_mean=("MAE","mean"),MAE_std=("MAE","std"),
    MAE_below_60_mean=("MAE_below_60","mean"),MAE_at_least_60_mean=("MAE_at_least_60","mean"),
    within_5_mean=("within_5_percent","mean"),p90_mean=("p90_abs_error","mean")).reset_index().sort_values("MAE_mean")
display(el_specialist_summary_df)

# Select the strategy on pre-test folds, not the historical test.
best_specialist=el_specialist_summary_df.iloc[0]
Xpre,Xte=_fold_features(pretest_idx,test_idx)
estimator=models[f"{best_specialist['model']}_physical"]
strategy=best_specialist["strategy"]
if strategy=="global":
    pred_specialist_test=clone(estimator).fit(Xpre,y_el[pretest_idx]).predict(Xte)
elif strategy=="hard_experts_60":
    pred_specialist_test=_specialist_prediction(estimator,Xpre,Xte,pretest_idx,test_idx,smooth=False)
elif strategy=="smooth_experts_55_65":
    pred_specialist_test=_specialist_prediction(estimator,Xpre,Xte,pretest_idx,test_idx,smooth=True)
else:
    high_weight=float(strategy.rsplit("_",1)[-1])
    weights=np.where(el_meta.iloc[pretest_idx]["Elevation"].to_numpy()>=60,high_weight,1.0)
    pred_specialist_test=_fit_weighted_pipeline(estimator,Xpre,y_el[pretest_idx],weights).predict(Xte)
err=np.abs(y_el[test_idx]-pred_specialist_test); high=el_meta.iloc[test_idx]["Elevation"].to_numpy()>=60
el_specialist_selected_test_df=pd.DataFrame([{
    "selected_model":best_specialist["model"],"selected_strategy":strategy,"cv_MAE_mean":best_specialist["MAE_mean"],
    "test_count":len(test_idx),"test_MAE":float(err.mean()),"test_MAE_below_60":float(err[~high].mean()),
    "test_MAE_at_least_60":float(err[high].mean()),"test_within_5_percent":float((err<=5).mean()*100),
    "test_p90_abs_error":float(np.quantile(err,.90))}])
display(el_specialist_selected_test_df)


In [ ]:
el_neural_results_df=pd.DataFrame()
if not TF_AVAILABLE:
    print('TensorFlow is unavailable; skipping EL neural tests.')
else:

    neural_imputer=SimpleImputer(strategy="median").fit(F_oracle_rx.iloc[train_idx])
    neural_scaler=StandardScaler().fit(neural_imputer.transform(F_oracle_rx.iloc[train_idx]))
    Xn_train=neural_scaler.transform(neural_imputer.transform(F_oracle_rx.iloc[train_idx])).astype("float32")
    Xn_val=neural_scaler.transform(neural_imputer.transform(F_oracle_rx.iloc[val_idx])).astype("float32")
    Xn_test=neural_scaler.transform(neural_imputer.transform(F_oracle_rx.iloc[test_idx])).astype("float32")
    y_pair=np.c_[y,y_el]
    center=np.median(y_pair[train_idx],axis=0); scale=np.quantile(y_pair[train_idx],.75,axis=0)-np.quantile(y_pair[train_idx],.25,axis=0)
    scale=np.where(scale>1e-6,scale,1.0)
    yp_train=((y_pair[train_idx]-center)/scale).astype("float32")
    yp_val=((y_pair[val_idx]-center)/scale).astype("float32")

    def build_shared_mlp(n_inputs,n_outputs,seed):
        tf.keras.backend.clear_session(); tf.random.set_seed(seed)
        inp=keras.Input(shape=(n_inputs,)); h=layers.Dense(48,activation="gelu",kernel_regularizer=keras.regularizers.l2(1e-3))(inp)
        h=layers.Dropout(.25)(h); h=layers.Dense(24,activation="gelu",kernel_regularizer=keras.regularizers.l2(1e-3))(h)
        out=layers.Dense(n_outputs,dtype="float32")(h)
        mdl=keras.Model(inp,out); mdl.compile(optimizer=keras.optimizers.AdamW(3e-4,weight_decay=1e-3),loss=keras.losses.Huber())
        return mdl

    def neural_callbacks():
        return [keras.callbacks.EarlyStopping(monitor="val_loss",patience=25,restore_best_weights=True),
                keras.callbacks.ReduceLROnPlateau(monitor="val_loss",patience=8,factor=.5,min_lr=1e-6)]


    neural_cv_rows=[]
    for fold,(tr_local,ev_local) in enumerate(TimeSeriesSplit(n_splits=4).split(pretest_idx),1):
        outer_tr,outer_ev=pretest_idx[tr_local],pretest_idx[ev_local]
        monitor_n=max(20,int(round(.2*len(outer_tr))))
        core_tr,monitor=outer_tr[:-monitor_n],outer_tr[-monitor_n:]
        Xcore,Xfuture=_fold_features(core_tr,np.r_[monitor,outer_ev])
        Xmon=Xfuture.iloc[:len(monitor)].reset_index(drop=True); Xouter=Xfuture.iloc[len(monitor):].reset_index(drop=True)
        imp=SimpleImputer(strategy="median").fit(Xcore); scl=StandardScaler().fit(imp.transform(Xcore))
        A=scl.transform(imp.transform(Xcore)).astype("float32")
        B=scl.transform(imp.transform(Xmon)).astype("float32")
        C=scl.transform(imp.transform(Xouter)).astype("float32")
        pair=np.c_[y,y_el]; ctr=np.median(pair[core_tr],axis=0)
        scl_y=np.quantile(pair[core_tr],.75,axis=0)-np.quantile(pair[core_tr],.25,axis=0); scl_y=np.where(scl_y>1e-6,scl_y,1)
        Ycore=((pair[core_tr]-ctr)/scl_y).astype("float32"); Ymon=((pair[monitor]-ctr)/scl_y).astype("float32")
        for architecture,n_outputs,tcore,tmon in [
            ("single_task_EL",1,Ycore[:,1:2],Ymon[:,1:2]),
            ("multi_task_XEL_EL",2,Ycore,Ymon),
        ]:
            mdl=build_shared_mlp(A.shape[1],n_outputs,RANDOM_SEED+100*fold+n_outputs)
            hist=mdl.fit(A,tcore,validation_data=(B,tmon),epochs=250,batch_size=24,verbose=0,callbacks=neural_callbacks())
            raw=mdl.predict(C,verbose=0); pred_el=raw[:,-1]*scl_y[1]+ctr[1]; err=np.abs(y_el[outer_ev]-pred_el)
            neural_cv_rows.append({"fold":fold,"architecture":architecture,"n_train":len(core_tr),"n_monitor":len(monitor),
                                   "n_eval":len(outer_ev),"epochs":len(hist.history["loss"]),"MAE":float(err.mean()),
                                   "within_5_percent":float((err<=5).mean()*100),"p90_abs_error":float(np.quantile(err,.90))})
    el_neural_temporal_cv_df=pd.DataFrame(neural_cv_rows)
    el_neural_temporal_summary_df=el_neural_temporal_cv_df.groupby("architecture",observed=True).agg(
        folds=("fold","nunique"),MAE_mean=("MAE","mean"),MAE_std=("MAE","std"),
        within_5_mean=("within_5_percent","mean"),p90_mean=("p90_abs_error","mean")).reset_index().sort_values("MAE_mean")
    display(el_neural_temporal_summary_df)

    neural_rows=[]

    for architecture,n_outputs,target_train,target_val in [
        ("single_task_EL",1,yp_train[:,1:2],yp_val[:,1:2]),
        ("multi_task_XEL_EL",2,yp_train,yp_val),
    ]:
        mdl=build_shared_mlp(Xn_train.shape[1],n_outputs,RANDOM_SEED+n_outputs)
        hist=mdl.fit(Xn_train,target_train,validation_data=(Xn_val,target_val),epochs=250,batch_size=24,verbose=0,callbacks=neural_callbacks())
        for split_name,Xev,idx in [("val",Xn_val,val_idx),("test",Xn_test,test_idx)]:
            raw=mdl.predict(Xev,verbose=0)
            pred_el=(raw[:,-1]*scale[1]+center[1]).ravel(); err=np.abs(y_el[idx]-pred_el)
            neural_rows.append({"architecture":architecture,"split":split_name,"count":len(idx),"epochs":len(hist.history["loss"]),
                                "MAE":float(err.mean()),"within_5_percent":float((err<=5).mean()*100),"p90_abs_error":float(np.quantile(err,.90))})

    # Blend two EL networks over their 55-65 degree overlap.
    low_tr=np.flatnonzero(el_meta.iloc[train_idx]["Elevation"].to_numpy()<=65)
    high_tr=np.flatnonzero(el_meta.iloc[train_idx]["Elevation"].to_numpy()>=55)
    low_val=np.flatnonzero(el_meta.iloc[val_idx]["Elevation"].to_numpy()<=65)
    high_val=np.flatnonzero(el_meta.iloc[val_idx]["Elevation"].to_numpy()>=55)
    specialist_models=[]
    for name,tr_loc,va_loc,seed in [("low",low_tr,low_val,RANDOM_SEED+11),("high",high_tr,high_val,RANDOM_SEED+12)]:
        mdl=build_shared_mlp(Xn_train.shape[1],1,seed)
        hist=mdl.fit(Xn_train[tr_loc],yp_train[tr_loc,1:2],validation_data=(Xn_val[va_loc],yp_val[va_loc,1:2]),
                     epochs=250,batch_size=20,verbose=0,callbacks=neural_callbacks())
        specialist_models.append(mdl)
    for split_name,Xev,idx in [("val",Xn_val,val_idx),("test",Xn_test,test_idx)]:
        plow=specialist_models[0].predict(Xev,verbose=0).ravel()*scale[1]+center[1]
        phigh=specialist_models[1].predict(Xev,verbose=0).ravel()*scale[1]+center[1]
        w=np.clip((el_meta.iloc[idx]["Elevation"].to_numpy()-55)/10,0,1)
        pred=(1-w)*plow+w*phigh; err=np.abs(y_el[idx]-pred)
        neural_rows.append({"architecture":"specialist_EL_55_65","split":split_name,"count":len(idx),"epochs":np.nan,
                            "MAE":float(err.mean()),"within_5_percent":float((err<=5).mean()*100),"p90_abs_error":float(np.quantile(err,.90))})
    el_neural_results_df=pd.DataFrame(neural_rows).sort_values(["split","MAE"])
    display(el_neural_results_df)


In [ ]:
exports = {
    "true_xel_projection_summary.csv": projection_summary_df,
    "true_xel_target_definition_ablation.csv": target_definition_ablation_df,
    "true_xel_receiver_ablation.csv": receiver_ablation_df,
    "true_xel_receiver_ablation_cv.csv": receiver_ablation_cv_df,
    "true_xel_performance_by_elevation.csv": performance_by_elevation_df,
    "true_xel_performance_by_receiver.csv": performance_by_receiver_df,
    "true_xel_receiver_fit_audit.csv": receiver_fit_audit_df,
    "true_xel_fem_sign_convention.csv": fem_sign_convention_df,
    "true_xel_raw_fem_corrected_target.csv": raw_fem_corrected_target_df,
}
if "repeatability_conditioned_df" in globals():
    exports["true_xel_repeatability_conditioned.csv"] = repeatability_conditioned_df
if "repeatability_by_receiver_df" in globals():
    exports["true_xel_repeatability_by_receiver.csv"] = repeatability_by_receiver_df
if len(gru_receiver_results_df):
    exports["true_xel_gru_receiver_ablation.csv"] = gru_receiver_results_df
if "moe_direct_results_df" in globals():
    exports["true_xel_moe_direct_results.csv"] = moe_direct_results_df
if "moe_receiver_ablation_df" in globals():
    exports["true_xel_moe_receiver_ablation.csv"] = moe_receiver_ablation_df
if "domain_importance_v2_df" in globals():
    exports["true_xel_domain_importance_v2.csv"] = domain_importance_v2_df
if "support_v2_summary" in globals():
    exports["true_xel_support_v2_summary.csv"] = support_v2_summary
if "conditional_v2_summary" in globals():
    exports["true_xel_conditional_v2_summary.csv"] = conditional_v2_summary
if "domain_ablation_df" in globals():
    exports["true_xel_domain_ablation.csv"] = domain_ablation_df
if "matched_support_ablation_df" in globals():
    exports["true_xel_matched_support_ablation.csv"] = matched_support_ablation_df
if "matched_conditional_shift_df" in globals():
    exports["true_xel_matched_conditional_shift.csv"] = matched_conditional_shift_df
if "matched_conditional_records_df" in globals():
    exports["true_xel_matched_conditional_records.csv"] = matched_conditional_records_df
if "conditional_by_support_df" in globals():
    exports["true_xel_conditional_by_support.csv"] = conditional_by_support_df
if "conditional_by_support_eval_df" in globals():
    exports["true_xel_conditional_by_support_eval.csv"] = conditional_by_support_eval_df
if "cluster_detail_df" in globals():
    exports["true_xel_june_25_27_cluster.csv"] = cluster_detail_df
if "cluster_context_df" in globals():
    exports["true_xel_june_25_27_context.csv"] = cluster_context_df
if "oracle_by_elevation_df" in globals():
    exports["true_xel_oracle_by_elevation.csv"] = oracle_by_elevation_df
if "oracle_by_receiver_df" in globals():
    exports["true_xel_oracle_by_receiver.csv"] = oracle_by_receiver_df
if "oracle_by_thermal_regime_df" in globals():
    exports["true_xel_oracle_by_thermal_regime.csv"] = oracle_by_thermal_regime_df
if "oracle_router_predictability_df" in globals():
    exports["true_xel_oracle_router_predictability.csv"] = oracle_router_predictability_df
if "selective_disagreement_curve_df" in globals():
    exports["true_xel_selective_disagreement_curve.csv"] = selective_disagreement_curve_df
if "selective_disagreement_threshold_df" in globals():
    exports["true_xel_selective_disagreement_threshold.csv"] = selective_disagreement_threshold_df
if "quality_thresholds_df" in globals():
    exports["true_xel_quality_thresholds.csv"] = quality_thresholds_df
if "quality_filter_summary_df" in globals():
    exports["true_xel_quality_filter_summary.csv"] = quality_filter_summary_df
if "quality_baseline_df" in globals():
    exports["true_xel_quality_baseline.csv"] = quality_baseline_df
if "quality_repeatability_df" in globals():
    exports["true_xel_quality_repeatability.csv"] = quality_repeatability_df
if "el_target_audit_df" in globals():
    exports["el_target_audit.csv"] = el_target_audit_df
if "el_target_distribution_df" in globals():
    exports["el_target_distribution.csv"] = el_target_distribution_df
if "el_repeatability_df" in globals():
    exports["el_repeatability.csv"] = el_repeatability_df
if "el_model_benchmark_df" in globals():
    exports["el_model_benchmark.csv"] = el_model_benchmark_df
if "el_selected_model_df" in globals():
    exports["el_selected_model.csv"] = el_selected_model_df
if "el_weighted_ridge_df" in globals():
    exports["el_weighted_ridge.csv"] = el_weighted_ridge_df
if "el_history_coverage_df" in globals():
    exports["el_history_coverage.csv"] = el_history_coverage_df
if "el_history_benchmark_df" in globals():
    exports["el_history_benchmark.csv"] = el_history_benchmark_df
if "el_history_common_test_df" in globals():
    exports["el_history_common_test.csv"] = el_history_common_test_df
if "el_residual_by_elevation_df" in globals():
    exports["el_residual_by_elevation.csv"] = el_residual_by_elevation_df
if "el_residual_by_azimuth_df" in globals():
    exports["el_residual_by_azimuth.csv"] = el_residual_by_azimuth_df
if "el_residual_by_receiver_df" in globals():
    exports["el_residual_by_receiver.csv"] = el_residual_by_receiver_df
if "el_residual_by_sun_thermal_df" in globals():
    exports["el_residual_by_sun_thermal.csv"] = el_residual_by_sun_thermal_df
if "el_quality_thresholds_df" in globals():
    exports["el_quality_thresholds.csv"] = el_quality_thresholds_df
if "el_quality_filter_summary_df" in globals():
    exports["el_quality_filter_summary.csv"] = el_quality_filter_summary_df
if "el_quality_benchmark_df" in globals():
    exports["el_quality_benchmark.csv"] = el_quality_benchmark_df
if "el_xel_residual_correlation_df" in globals():
    exports["el_xel_residual_correlation.csv"] = el_xel_residual_correlation_df
if "el_xel_stacking_cv_df" in globals():
    exports["el_xel_stacking_cv.csv"] = el_xel_stacking_cv_df
if "el_xel_stacking_summary_df" in globals():
    exports["el_xel_stacking_summary.csv"] = el_xel_stacking_summary_df
if "el_multiscale_cv_df" in globals():
    exports["el_multiscale_cv.csv"] = el_multiscale_cv_df
if "el_multiscale_summary_df" in globals():
    exports["el_multiscale_summary.csv"] = el_multiscale_summary_df
if "el_specialist_cv_df" in globals():
    exports["el_specialist_cv.csv"] = el_specialist_cv_df
if "el_specialist_summary_df" in globals():
    exports["el_specialist_summary.csv"] = el_specialist_summary_df
if "el_specialist_selected_test_df" in globals():
    exports["el_specialist_selected_test.csv"] = el_specialist_selected_test_df
if "el_neural_results_df" in globals() and len(el_neural_results_df):
    exports["el_neural_multitask_specialists.csv"] = el_neural_results_df
if "el_neural_temporal_cv_df" in globals():
    exports["el_neural_temporal_cv.csv"] = el_neural_temporal_cv_df
if "el_neural_temporal_summary_df" in globals():
    exports["el_neural_temporal_summary.csv"] = el_neural_temporal_summary_df

for filename, frame in exports.items():
    frame.to_csv(filename, index=False)
print('Exports completed:')
for filename in exports:
    print(" -", filename)
